# Fractal Image Compression — GPU Accelerated v7.3

## Changelog

**v7.3 — Batched GPU encoder + research validation cells**
- `encode_channel_gpu` rewritten: Python per-block loop replaced with a single
  batched matrix operation `R @ ds.T` across all M range blocks simultaneously.
  Eliminates O(M) GPU kernel launches and O(M) host→GPU transfers. OOM-safe via
  automatic chunking.
- Added Cell D: Kodak-24 dataset evaluation (backs sample-size critique)
- Added Cell E: Threshold multiplier sweep (backs τ = 0.50 × median_var)
- Added Cell F: Residual gate justification (backs 40–70% gate across all images)
- Fixed `encode_colour_channel` call sites: `cfg` dict → `DCT_QUALITY_FACTOR` float

**v7.2 — Portrait image + upscaling + JPEG comparison cells**

**v7.1 — 5-bit cy overflow fix** (expanded to 6-bit, supports images up to ~1528px at step=24)

**v7 — Core fixes**
- Variance grouping removed (was broken: grouped domain variance not range variance)
- Brightness stored as uint8 [0,255] not int8 (fixed systematic underexposure for Y>200)
- Colour encoder: pickle.dumps replaced with compact binary DCT + zlib

---

## Notebook structure

| Section | Cells | Purpose |
|---------|-------|---------|
| A — Core functions | 1–10 | All functions needed to run the pipeline |
| B — Main pipeline | 11 | Encode/decode a single image |
| C — Benchmark cells | 12–14 | Upscaling comparison, JPEG comparison, rate-distortion |
| D — Research cells | 15–17 | Dataset evaluation, threshold sweep, residual gate justification |


## A1 — Setup

Install dependencies and confirm GPU availability. Run once per Colab session.

In [5]:
!pip install cupy-cuda12x scipy -q
!nvidia-smi
from IPython.display import display


Wed Apr  8 15:03:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             31W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## A2 — Imports and GPU Detection

CuPy is used as a drop-in NumPy replacement for GPU arrays. If CuPy is
unavailable (no GPU), the code falls back to NumPy automatically — results
will be identical but slower.

In [6]:
import numpy as np
import math, time, struct, os, random, zlib, io, json
from PIL import Image
from scipy.fft import dctn, idctn
import matplotlib.pyplot as plt

try:
    import cupy as cp
    cp.cuda.Device(0).compute_capability
    xp = cp
    print('GPU ready (CuPy)')
except Exception as e:
    xp = np
    print(f'CPU fallback: {e}')


GPU ready (CuPy)


## A3 — Configuration

**Constants (do not change):**
- `BLOCK_SIZE=8, DOMAIN_SIZE=16`: standard PIFS block sizes; domain is 2× range
  so downsampling by 2 is exact.
- `NUM_TRANSFORMS=8`: D4 symmetry group (4 rotations × 2 flips).
- `N_ITERATIONS=10`: IFS decode iterations; convergence is stable by iteration 7–8.

**Threshold:** `τ = 0.50 × median_var` — calibrates the early-stop classifier
to the image's own variance distribution. The 0.50 constant makes τ independent
of absolute image brightness. Note: the encoder always stores the globally optimal
match for every block; τ only classifies whether that match was "good enough"
for the early-stop rate counter. PSNR is invariant to τ for a fixed domain pool.

**Residual gate:** the second fractal pass runs only when `40% < early_stop_rate < 70%`.
Outside this range: below 40% the error is a diffuse gradient that fractal cannot
encode; above 70% the first pass already found excellent matches and the residual
is dominated by noise. Gate boundaries are validated by Cell F across Kodak-24.

**File paths:** set `IMAGE_PATH` and `OUTPUT_PATH` before running Section B.

In [7]:
BLOCK_SIZE     = 8
DOMAIN_SIZE    = 16
NUM_TRANSFORMS = 8
N_ITERATIONS   = 10

COMPRESSION_MODE   = 'LOSSY'   # 'LOSSLESS' or 'LOSSY'
DCT_QUALITY_FACTOR = 1.0       # LOSSY: lower = larger file, less colour loss

ENCODE_RESIDUAL    = True      # master switch (True = activate if gate conditions met)
RESIDUAL_MIN_EARLY = 0.40      # below: error is diffuse gradient, residual unhelpful
RESIDUAL_MAX_EARLY = 0.70      # above: error is random noise, residual adds artifacts

AUTO_THRESHOLD      = True     # derive τ from image variance automatically
TIME_BUDGET_SECONDS = 120      # used to auto-select domain step
DOMAIN_STEP         = 32       # manual fallback if AUTO_THRESHOLD = False
ERROR_THRESHOLD     = 50.0     # manual fallback if AUTO_THRESHOLD = False

IMAGE_PATH  = '/content/your_image.png'   # ← set this
OUTPUT_PATH = '/content/output.frac'      # ← set this
ENCODE      = True                        # True = encode; False = load from OUTPUT_PATH

# Standard JPEG chroma quantisation table (used for Cb/Cr DCT encoding)
CHROMA_QUANT_BASE = np.array([
    [16,11,10,16, 24, 40, 51, 61],
    [12,12,14,19, 26, 58, 60, 55],
    [14,13,16,24, 40, 57, 69, 56],
    [14,17,22,29, 51, 87, 80, 62],
    [18,22,37,56, 68,109,103, 77],
    [24,35,55,64, 81,104,113, 92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103, 99],
], dtype=np.float32)

def validate_domain_step(H_pad, W_pad, step):
    """
    Confirm that domain grid fits within 6-bit coordinate fields (max index 63).
    The 6-bit field stores the domain block index, not a pixel coordinate.
    At step=24 this covers images up to ~1528 pixels per side.
    """
    cy_max = (H_pad - DOMAIN_SIZE) // step
    cx_max = (W_pad - DOMAIN_SIZE) // step
    if cy_max > 63:
        raise ValueError(
            f'cy_max={cy_max} overflows 6-bit field. '
            f'Min safe step={math.ceil((H_pad-DOMAIN_SIZE)/63)}')
    if cx_max > 63:
        raise ValueError(f'cx_max={cx_max} overflows 6-bit field')

def compute_auto_domain_step(H_pad, W_pad, budget_sec):
    """
    Select domain step to keep estimated encode time within budget_sec.
    Smaller step = denser domain pool = better quality = slower.
    """
    n_blocks  = (H_pad // BLOCK_SIZE) * (W_pad // BLOCK_SIZE)
    max_cand  = budget_sec / (n_blocks * 2.5e-8)
    step = max(8, min(64, int(math.ceil(
        math.sqrt((H_pad * W_pad * 8) / max(max_cand, 1)) / 8)) * 8))
    while True:
        try: validate_domain_step(H_pad, W_pad, step); break
        except: step += 8
        if step > 256: break
    return step

print('Configuration defined')


Configuration defined


## A4 — Colour Space and Padding

YCbCr separates luminance (Y) from chroma (Cb, Cr). Fractal encoding operates
on Y only; chroma is handled by the compact DCT codec. This exploits the human
visual system's lower sensitivity to colour detail (chroma subsampling principle).

Padding to 8-pixel multiples uses edge replication to avoid introducing
artificial boundaries. Original dimensions are stored in the .frac header
for exact cropping on decode.

In [8]:
def rgb_to_ycbcr(image):
    img = image.astype(np.float32)
    m   = np.array([[ 0.299,  0.587,  0.114],
                    [-0.169, -0.331,  0.500],
                    [ 0.500, -0.419, -0.081]])
    y = img @ m.T
    y[:,:,1] += 128; y[:,:,2] += 128
    return np.clip(y, 0, 255).astype(np.uint8)

def ycbcr_to_rgb(image):
    img = image.astype(np.float32)
    img[:,:,1] -= 128; img[:,:,2] -= 128
    inv = np.array([[1.000,  0.000,  1.402],
                    [1.000, -0.344, -0.714],
                    [1.000,  1.772,  0.000]])
    return np.clip(img @ inv.T, 0, 255).astype(np.uint8)

def pad_to_block_multiple(img):
    H, W = img.shape[:2]
    Hp = math.ceil(H / BLOCK_SIZE) * BLOCK_SIZE
    Wp = math.ceil(W / BLOCK_SIZE) * BLOCK_SIZE
    if Hp == H and Wp == W:
        return img, H, W
    print(f'  Padded {W}x{H} -> {Wp}x{Hp}')
    return np.pad(img, ((0,Hp-H),(0,Wp-W),(0,0)), mode='edge'), H, W

def crop_to_original(img, H, W):
    return img[:H, :W]

print('Colour transforms defined')


Colour transforms defined


## A5 — Fractal Helpers

`downsample_2x`: exact 2× area average — each 2×2 pixel group becomes one pixel.
Used to produce domain blocks from 16×16 regions.

`get_all_transforms`: applies all 8 elements of the dihedral group D4 to a block
(4 rotations × 2 reflections). This gives the encoder orientation-invariant
matching without storing the transform orientation explicitly — only the index
0–7 is stored in the .frac file.

In [9]:
def downsample_2x(block):
    h, w = block.shape
    return block.reshape(h//2, 2, w//2, 2).mean(axis=(1,3)).astype(np.float32)

def get_all_transforms(block):
    """Return all 8 D4 variants of an 8x8 block (4 rotations x 2 flips)."""
    variants = []
    for flip in [False, True]:
        b = np.fliplr(block) if flip else block.copy()
        for r in range(4):
            variants.append(np.rot90(b, r).copy())
    return np.array(variants, dtype=np.float32)

print('Fractal helpers defined')


Fractal helpers defined


## A6 — Image Analysis and Adaptive Parameter Selection

Computes per-block variance across the Y channel to derive two parameters:

**Error threshold τ:** set to `clip(0.50 × median_var, 5, 500)`. The 0.50
multiplier calibrates τ relative to the image's own complexity. Because the
encoder always finds the globally optimal domain match (argmin over all N
candidates), τ does not affect per-block quality — it only classifies whether
the optimal match was "good enough" for the early-stop rate counter. Across
21 Kodak images, this calibration consistently placed the early-stop rate in
the 51–70% range regardless of image content class.

**Domain step:** selected from image dimensions and time budget so encode
completes within the budget. Smaller step = more candidates = denser search.
High-variance images (e.g. foliage) receive a finer step automatically.

In [10]:
def analyse_image(channel, time_budget_sec=120, auto_threshold=True):
    H, W = channel.shape
    ch   = channel.astype(np.float32)
    Hp   = math.ceil(H / BLOCK_SIZE) * BLOCK_SIZE
    Wp   = math.ceil(W / BLOCK_SIZE) * BLOCK_SIZE
    print('  Analysing image...')

    range_vars = np.array([
        ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].var()
        for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)
    ])
    total_blocks = len(range_vars)

    # Flat detection: bottom 5th percentile, capped at variance=50
    FLAT_THRESH = min(float(np.percentile(range_vars, 5)), 50.0)
    flat_mask   = range_vars < FLAT_THRESH
    flat_means  = {}
    bi = 0
    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            if flat_mask[bi]:
                flat_means[(ry, rx)] = float(
                    ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].mean())
            bi += 1
    print(f'  Flat blocks: {flat_mask.sum()}/{total_blocks} '
          f'({flat_mask.sum()/total_blocks*100:.1f}%)')

    complex_vars = range_vars[~flat_mask]
    median_var   = float(np.median(complex_vars)) if len(complex_vars) > 0 else 100.0

    if auto_threshold:
        auto_thresh = float(np.clip(median_var * 0.50, 5.0, 500.0))
        print(f'  Median block variance: {median_var:.1f}')
        print(f'  Auto ERROR_THRESHOLD:  {auto_thresh:.1f}')
    else:
        auto_thresh = ERROR_THRESHOLD

    auto_step = compute_auto_domain_step(Hp, Wp, time_budget_sec)
    print(f'  Auto DOMAIN_STEP: {auto_step}  (budget={time_budget_sec}s)')

    return {
        'error_threshold': auto_thresh,
        'domain_step':     auto_step,
        'flat_mask':       flat_mask,
        'flat_means':      flat_means,
        'flat_pct':        flat_mask.sum() / total_blocks * 100,
        'median_var':      median_var,
        'total_blocks':    total_blocks,
        'range_vars':      range_vars,
    }

print('Image analysis defined')


Image analysis defined


## A7 — Pre-Sample Diagnostic

Encodes a stratified 5% sample of blocks to estimate the full-encode early-stop
rate before committing to a full encode. Uses three variance buckets (low/mid/high)
to ensure the sample represents all content types in the image.

**What early-stop rate predicts:** search success — whether the domain pool
contains good matches. For smooth-content images this correlates directly with
reconstruction quality. For high-entropy images (foliage), a 50% early-stop rate
can still produce low PSNR because the domain pool contains structurally plausible
but texturally incorrect matches (attractor mismatch, not search failure).

Across all 5 benchmark images and 21 Kodak images, sample rates matched full
encode rates within 3.1 percentage points.

In [11]:
def run_presample_diagnostic(channel, domain_stack, cfg, sample_pct=0.05):
    H, W       = channel.shape
    ch         = channel.astype(np.float32)
    thresh     = cfg['error_threshold']
    flat_mask  = cfg['flat_mask']
    range_vars = cfg['range_vars']

    all_pos = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]

    # Stratified: equal samples from low / mid / high variance buckets
    p33 = float(np.percentile(range_vars, 33))
    p66 = float(np.percentile(range_vars, 66))
    buckets = [[], [], []]
    for i, (ry, rx) in enumerate(all_pos):
        if flat_mask[i]: continue
        g = 0 if range_vars[i] <= p33 else (1 if range_vars[i] <= p66 else 2)
        buckets[g].append((ry, rx))

    n_each = max(5, int(len([p for b in buckets for p in b]) * sample_pct / 3))
    sample  = []
    for b in buckets:
        sample += random.sample(b, min(n_each, len(b)))

    print(f'  Pre-sample: {len(sample)} blocks stratified across 3 variance groups...')
    t0 = time.time()

    n_px   = float(BLOCK_SIZE * BLOCK_SIZE)
    sum_d  = xp.sum(domain_stack, axis=1)
    sum_d2 = xp.sum(domain_stack ** 2, axis=1)
    denom  = n_px * sum_d2 - sum_d ** 2
    flat_d = xp.abs(denom) < 1e-6

    early = 0
    best_errors = []
    for ry, rx in sample:
        rf     = xp.array(ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].flatten())
        sum_r  = xp.sum(rf)
        sum_dr = domain_stack @ rf
        cq = xp.clip((n_px*sum_dr - sum_d*sum_r) / (denom+1e-10), -0.75, 0.75)
        bq = xp.clip((sum_r - cq*sum_d) / n_px, 0.0, 255.0)
        cq = xp.where(flat_d, 0.0, cq)
        bq = xp.where(flat_d, sum_r / n_px, bq)
        approx = domain_stack * cq[:,None] + bq[:,None]
        errors = xp.mean((approx - rf[None,:]) ** 2, axis=1)
        best_e = float(xp.min(errors))
        best_errors.append(best_e)
        if best_e <= thresh: early += 1

    elapsed    = time.time() - t0
    early_pct  = early / len(sample)
    median_err = float(np.median(best_errors))
    print(f'  Early-stop rate: {early_pct*100:.1f}%  '
          f'median_best_err={median_err:.1f}  thresh={thresh:.1f}')
    print(f'  Sample: {elapsed:.1f}s  full estimate: {elapsed/sample_pct:.0f}s')

    refined = dict(cfg)
    refined['early_stop_pct'] = early_pct

    if early_pct < 0.10:
        new_t = float(np.clip(median_err * 0.9, thresh, 500.0))
        refined['error_threshold'] = new_t
        refined['predicted_quality'] = 'POOR'
        print(f'  WARNING: canopy-class ({early_pct*100:.1f}% early). '
              f'Threshold raised {thresh:.1f}->{new_t:.1f}')
    elif early_pct > 0.90:
        new_t = float(np.clip(median_err * 1.5, 5.0, thresh))
        refined['error_threshold'] = new_t
        refined['predicted_quality'] = 'EXCELLENT'
        print(f'  Excellent self-similarity. Tightening threshold to {new_t:.1f}')
    elif early_pct > 0.60:
        refined['predicted_quality'] = 'GOOD'
        print(f'  Good self-similarity.')
    else:
        refined['predicted_quality'] = 'MODERATE'
        print(f'  Moderate self-similarity.')

    return refined

print('Pre-sample diagnostic defined')


Pre-sample diagnostic defined


## A8 — Domain Stack and Batched GPU Encoder (v7.3)

### build_domain_stack
Constructs the (N, 64) domain candidate matrix. For each domain position,
all 8 D4 transforms are applied and flattened. This is done in NumPy then
transferred to GPU as a single array — one transfer for the whole stack.

### encode_channel_gpu — v7.3 batched redesign

**v7.2 (old):** Python for-loop over M range blocks. Each iteration:
one `xp.array()` host→GPU transfer, one `ds @ rf` matmul. O(M) GPU kernel launches.

**v7.3 (new):** all M non-flat range blocks stacked into R ∈ ℝ^(M×64)
and transferred in a **single** host→GPU call. The full (M,N) error surface
is computed analytically from **one** batched matmul `R @ ds.T` without
materialising the (M,N,64) error tensor:

```
MSE(m,n) = cq²·mean(d²) + bq² + mean(r²) + 2·cq·bq·mean(d) − 2·cq·mean(dr) − 2·bq·mean(r)
```

argmin over N gives the best domain for all M blocks simultaneously.
OOM-safe: if GPU memory is insufficient the chunk size halves automatically.

In [12]:
def build_domain_stack(channel, domain_step):
    """
    Build domain candidate array D ∈ ℝ^(N, 64).
    N = num_positions × 8 (D4 transforms).
    All transforms computed in NumPy; transferred to GPU as one block.
    """
    H, W = channel.shape
    candidates, meta = [], []
    for dy in range(0, H-DOMAIN_SIZE+1, domain_step):
        for dx in range(0, W-DOMAIN_SIZE+1, domain_step):
            raw = channel[dy:dy+DOMAIN_SIZE, dx:dx+DOMAIN_SIZE]
            ds  = downsample_2x(raw)
            tfs = get_all_transforms(ds)
            for ti in range(NUM_TRANSFORMS):
                candidates.append(tfs[ti].flatten())
                meta.append((dy, dx, ti))
    dg = xp.array(np.array(candidates, dtype=np.float32))
    print(f'    {dg.shape[0]:,} candidates')
    return dg, meta


def encode_channel_gpu(channel, cfg):
    """
    v7.3 batched GPU encoder.
    Returns (transforms list, early_stop_pct).
    Interface identical to v7.2 — no other cells require changes.
    """
    H, W       = channel.shape
    ch         = channel.astype(np.float32)
    thresh     = cfg['error_threshold']
    step       = cfg['domain_step']
    flat_means = cfg['flat_means']

    t0 = time.time()
    ds, meta = build_domain_stack(ch, step)   # ds: (N, 64) on GPU
    N = ds.shape[0]
    print(f'    {N:,} candidates in {time.time()-t0:.2f}s')

    n_px = float(BLOCK_SIZE * BLOCK_SIZE)   # 64.0

    # Domain statistics — computed once, reused across all range blocks
    sum_d  = xp.sum(ds, axis=1)            # (N,)
    sum_d2 = xp.sum(ds ** 2, axis=1)       # (N,)
    denom  = n_px * sum_d2 - sum_d ** 2    # (N,) denominator for least-squares
    flat_d = xp.abs(denom) < 1e-6          # (N,) marks uniform/flat domain blocks

    all_pos = [
        (ry, rx)
        for ry in range(0, H - BLOCK_SIZE + 1, BLOCK_SIZE)
        for rx in range(0, W - BLOCK_SIZE + 1, BLOCK_SIZE)
    ]
    total_blocks = len(all_pos)

    # Separate flat bypass blocks from blocks requiring domain search
    flat_results = {}
    search_pos   = []
    for bi, (ry, rx) in enumerate(all_pos):
        if (ry, rx) in flat_means:
            flat_results[(ry, rx)] = (0, 0, 0, 0.0, flat_means[(ry, rx)])
        else:
            search_pos.append((ry, rx))

    M = len(search_pos)
    print(f'    Batched search: {M} non-flat blocks × {N} candidates')

    # Stack all range blocks into R ∈ ℝ^(M, 64) — single host→GPU transfer
    R_np = np.stack([
        ch[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].flatten()
        for ry, rx in search_pos
    ], axis=0).astype(np.float32)
    R = xp.array(R_np)

    sum_r  = xp.sum(R, axis=1)             # (M,)
    sum_r2 = xp.sum(R ** 2, axis=1)        # (M,)

    # Chunked batched solve — halves chunk on OOM
    CHUNK = M
    best_idx_all = xp.zeros(M, dtype=xp.int32)
    best_err_all = xp.full(M, xp.inf, dtype=xp.float32)
    best_cq_all  = xp.zeros(M, dtype=xp.float32)
    best_bq_all  = xp.zeros(M, dtype=xp.float32)

    offset = 0
    while offset < M:
        chunk_ok = False
        while not chunk_ok:
            try:
                end = min(offset + CHUNK, M)
                Rc  = R[offset:end]                          # (c, 64)
                src = sum_r[offset:end]                      # (c,)
                c   = end - offset

                # Key operation: one matmul replaces c individual matmuls
                sum_dr = Rc @ ds.T                           # (c, N)

                # Analytical least-squares solution for all (range, domain) pairs
                cq_raw = (n_px * sum_dr - sum_d[None,:] * src[:,None])                          / (denom[None,:] + 1e-10)           # (c, N)
                cq = xp.clip(cq_raw, -0.75, 0.75)           # contractivity constraint

                bq = (src[:,None] - cq * sum_d[None,:]) / n_px  # (c, N)

                # Override flat domains
                cq[:, flat_d] = 0.0
                bq_mean = (src / n_px)[:,None] * xp.ones(
                    (c, int(flat_d.sum())), dtype=xp.float32) if flat_d.any() else bq[:,flat_d]
                bq[:, flat_d] = bq_mean

                # Analytical MSE — avoids materialising (c, N, 64) tensor
                errors = (
                    cq**2 * (sum_d2[None,:] / n_px)
                    + bq**2
                    + (sum_r2[offset:end,None] / n_px)
                    + 2.0 * cq * bq * (sum_d[None,:] / n_px)
                    - 2.0 * cq * (sum_dr / n_px)
                    - 2.0 * bq * (src[:,None] / n_px)
                )                                            # (c, N)

                bidx = xp.argmin(errors, axis=1)            # (c,)
                berr = errors[xp.arange(c), bidx]
                bcq  = cq[xp.arange(c), bidx]
                bbq  = bq[xp.arange(c), bidx]

                best_idx_all[offset:end] = bidx
                best_err_all[offset:end] = berr
                best_cq_all [offset:end] = bcq
                best_bq_all [offset:end] = bbq

                offset   += CHUNK
                chunk_ok  = True

            except Exception as e:
                if 'memory' in str(e).lower() or 'alloc' in str(e).lower():
                    CHUNK = max(64, CHUNK // 2)
                    print(f'    OOM — retrying with chunk={CHUNK}')
                else:
                    raise

    # Pull results back to CPU
    to_np = lambda x: x.get() if hasattr(x, 'get') else np.array(x)
    bidx_cpu = to_np(best_idx_all)
    berr_cpu = to_np(best_err_all)
    bcq_cpu  = to_np(best_cq_all)
    bbq_cpu  = to_np(best_bq_all)

    # Reconstruct full transform list in original block order
    transforms = []
    early = 0
    si    = 0

    start = time.time()
    for bi, (ry, rx) in enumerate(all_pos):
        if (ry, rx) in flat_results:
            transforms.append(flat_results[(ry, rx)])
            early += 1
        else:
            dy, dx, ti = meta[bidx_cpu[si]]
            transforms.append((dy, dx, ti, float(bcq_cpu[si]), float(bbq_cpu[si])))
            if berr_cpu[si] <= thresh: early += 1
            si += 1

    elapsed   = time.time() - t0
    flat_count = len(flat_results)
    nf         = total_blocks - flat_count
    ep         = (early - flat_count) / max(nf, 1)
    print(f'\n    Done {elapsed:.1f}s  flat={flat_count}/{total_blocks}  '
          f'early={early-flat_count}/{nf} ({ep*100:.1f}%)')
    return transforms, ep

print('Batched GPU encoder v7.3 defined')


Batched GPU encoder v7.3 defined


## A9 — Fractal Decoder

Applies the stored IFS transforms iteratively from a flat-grey initial image.
Convergence is guaranteed by the contractivity constraint on c (∈ [−0.75, 0.75]).
Standard decode uses `n_iterations=10`; the upscale decoder uses 12 for stability
on larger canvases.

The decoder is intentionally CPU-only (NumPy) — it is called once per image and
is not the bottleneck. GPU decode would complicate the upscaling logic with minimal
practical benefit.

In [13]:
def decode_fractal(transforms, image_shape, n_iterations=10):
    H, W    = image_shape
    current = np.full((H, W), 128.0, dtype=np.float32)
    pos     = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]
    for it in range(n_iterations):
        nxt = np.zeros((H, W), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = transforms[bi]
            raw    = current[dy:dy+DOMAIN_SIZE, dx:dx+DOMAIN_SIZE]
            domain = downsample_2x(raw)
            if ti >= 4: domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE] = c * domain + b
        current = nxt
        print(f'    Iteration {it+1}/{n_iterations}', end='\r')
    print()
    return np.clip(current, 0, 255).astype(np.uint8)


def decode_fractal_upscaled(transforms, original_shape, scale=2, n_iter=12):
    """
    Decode at scale× by applying transforms on a proportionally larger canvas.
    The stored transforms are scale-invariant: applying the same rules on a
    larger grid produces a higher-resolution output from the same .frac file.
    Quality converges to a stable floor by 4× on smooth-content images.
    """
    ratio   = DOMAIN_SIZE // BLOCK_SIZE
    H, W    = original_shape
    sH, sW  = H*scale, W*scale
    sB, sD  = BLOCK_SIZE*scale, DOMAIN_SIZE*scale
    current = np.full((sH, sW), 128.0, dtype=np.float32)
    pos     = [(ry, rx)
               for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE)
               for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE)]
    for it in range(n_iter):
        nxt = np.zeros((sH, sW), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = transforms[bi]
            sry, srx = ry*scale, rx*scale
            sdy, sdx = dy*scale, dx*scale
            raw = current[sdy:sdy+sD, sdx:sdx+sD]
            if raw.shape[0] < sD or raw.shape[1] < sD:
                nxt[sry:sry+sB, srx:srx+sB] = b; continue
            domain = raw.reshape(sB, ratio, sB, ratio).mean(axis=(1,3))
            if ti >= 4: domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[sry:sry+sB, srx:srx+sB] = c*domain + b
        current = nxt
        print(f'    Upscale iter {it+1}/{n_iter}', end='\r')
    print()
    return np.clip(current, 0, 255).astype(np.uint8)

def decode_and_upscale(y_tf, cb_enc, cr_enc, padded_shape, orig_shape, scale=2, n_i=12):
    H, W = padded_shape; oH, oW = orig_shape
    y_up = decode_fractal_upscaled(y_tf, (H,W), scale=scale, n_iter=n_i)
    cb   = decode_colour_channel(cb_enc)
    cr   = decode_colour_channel(cr_enc)
    cb_up = np.array(Image.fromarray(cb).resize(
        (W*scale, H*scale), Image.BILINEAR))
    cr_up = np.array(Image.fromarray(cr).resize(
        (W*scale, H*scale), Image.BILINEAR))
    rgb = ycbcr_to_rgb(np.stack([y_up, cb_up, cr_up], axis=2))
    return rgb[:oH*scale, :oW*scale]

print('Decoders defined')


Decoders defined


## A10 — Residual Encoding (Gated)

A second fractal pass encodes the error signal (original − first-pass reconstruction),
shifted into [0, 255] to handle negative values. The gate activates when
`40% < early_stop_pct < 70%`.

**Why gated:**
- **Below 40%:** the image has too little self-similarity for fractal to encode the
  error effectively. The residual is a diffuse gradient — adding a second pass wastes
  file size with negligible quality gain.
- **Above 70%:** the first pass already found excellent matches. The remaining error
  is close to the noise floor of the domain approximation; a second fractal pass
  approximates this noise poorly and can introduce block artifacts that slightly
  reduce PSNR.
- **Between 40–70%:** enough structured residual exists to improve, and the domain
  pool has sufficient coverage to encode it. Gate boundaries validated by Cell F
  across 21 Kodak images.

In [14]:
def encode_residual(original_y, reconstructed_y, cfg):
    orig    = original_y.astype(np.float32)
    recon   = reconstructed_y.astype(np.float32)
    error   = orig - recon
    shifted = np.clip(error + 128, 0, 255).astype(np.uint8)  # shift to [0,255]

    mse_before = float(np.mean(error ** 2))
    print(f'  Residual MSE before pass 2: {mse_before:.2f}')
    print(f'  Error range: {error.min():.0f} to {error.max():.0f}  '
          f'Shifted: {shifted.min()}-{shifted.max()}')

    res_cfg = analyse_image(shifted,
                            time_budget_sec=TIME_BUDGET_SECONDS // 2,
                            auto_threshold=True)
    res_cfg['domain_step'] = cfg['domain_step']
    res_transforms, _ = encode_channel_gpu(shifted, res_cfg)
    return res_transforms


def decode_with_residual(pass1_transforms, pass2_transforms, image_shape, n_iter=10):
    print('  Decoding pass 1 (main)...')
    y1 = decode_fractal(pass1_transforms, image_shape, n_iter).astype(np.float32)
    print('  Decoding pass 2 (residual)...')
    y2 = decode_fractal(pass2_transforms, image_shape, n_iter).astype(np.float32)
    combined = np.clip(y1 + (y2 - 128.0), 0, 255).astype(np.uint8)
    print(f'  Combined: {combined.min()}-{combined.max()}')
    return combined

print('Residual encoding defined (gated 40-70%)')


Residual encoding defined (gated 40-70%)


## A11 — Compact Binary DCT Colour Encoder (v7)

Replaces the v6 `pickle.dumps` approach which added ~28× serialisation overhead.

**Format per 8×8 block:**
1. `n` (1 byte): number of non-trailing-zero coefficients in zigzag order (0–64)
2. `n × 2 bytes`: the n int16 zigzag coefficients

The entire stream is zlib-compressed at level 6. At 93–100% DCT zero coefficients
(typical for natural image chroma), each block compresses to 1–5 bytes.
Total colour streams: 4–14 KB per image. Greyscale images (e.g. aerial CCTV footage)
produce 100% zero coefficients → 0 KB colour storage.

In [15]:
def _zigzag_order(n=8):
    idx = []
    for d in range(2*n - 1):
        if d % 2 == 0:
            r = min(d, n-1); c = d - r
            while r >= 0 and c < n: idx.append((r, c)); r -= 1; c += 1
        else:
            c = min(d, n-1); r = d - c
            while c >= 0 and r < n: idx.append((r, c)); r += 1; c -= 1
    return idx

_ZZ = _zigzag_order()

def encode_channel_dct_compact(channel, quality_factor=1.0):
    H, W = channel.shape
    qt   = np.clip(CHROMA_QUANT_BASE * quality_factor, 1, 255)
    buf  = bytearray()
    total_zeros = 0
    n_blocks    = 0

    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            block  = channel[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE].astype(np.float32) - 128
            coeffs = dctn(block, norm='ortho')
            quant  = np.round(coeffs / qt).astype(np.int16)
            zz     = np.array([quant[r, c] for r, c in _ZZ], dtype=np.int16)
            last_nz = 0
            for k in range(63, -1, -1):
                if zz[k] != 0: last_nz = k + 1; break
            total_zeros += (64 - last_nz)
            n_blocks    += 1
            buf += bytes([last_nz]) + zz[:last_nz].tobytes()

    zero_pct   = total_zeros / (n_blocks * 64) * 100
    compressed = zlib.compress(bytes(buf), level=6)
    print(f'      DCT zero coeff: {zero_pct:.1f}%  '
          f'raw={len(buf)/1024:.0f}KB  compressed={len(compressed)/1024:.0f}KB')
    return compressed, qt, (H, W)


def decode_channel_dct_compact(compressed, quant_table, original_shape):
    H, W = original_shape
    data = zlib.decompress(compressed)
    out  = np.zeros((H, W), dtype=np.float32)
    i    = 0
    for ry in range(0, H-BLOCK_SIZE+1, BLOCK_SIZE):
        for rx in range(0, W-BLOCK_SIZE+1, BLOCK_SIZE):
            last_nz = data[i]; i += 1
            zz = np.zeros(64, dtype=np.float32)
            if last_nz > 0:
                vals = np.frombuffer(data[i:i+last_nz*2],
                                     dtype=np.int16).astype(np.float32)
                zz[:last_nz] = vals
                i += last_nz * 2
            grid = np.zeros((8, 8), dtype=np.float32)
            for k, (r, c) in enumerate(_ZZ):
                grid[r, c] = zz[k] * quant_table[r, c]
            out[ry:ry+BLOCK_SIZE, rx:rx+BLOCK_SIZE] = idctn(grid, norm='ortho') + 128
    return np.clip(out, 0, 255).astype(np.uint8)


def _delta_encode(ch):
    d = np.empty_like(ch, dtype=np.uint8)
    d[:,0] = ch[:,0]
    d[:,1:] = ((ch[:,1:].astype(np.int16) - ch[:,:-1].astype(np.int16)) % 256).astype(np.uint8)
    return d

def _delta_decode(d):
    s = d.astype(np.int32)
    s[:,1:] = np.where(d[:,1:]>127, d[:,1:].astype(np.int32)-256, d[:,1:].astype(np.int32))
    return np.clip(np.cumsum(s, axis=1) % 256, 0, 255).astype(np.uint8)

def encode_channel_lossless(channel):
    H, W  = channel.shape
    delta = _delta_encode(channel)
    comp  = zlib.compress(delta.tobytes(), level=6)
    print(f'      Lossless: {channel.size/1024:.0f}KB -> {len(comp)/1024:.0f}KB')
    return comp, (H, W)

def decode_channel_lossless(comp_bytes, original_shape):
    H, W  = original_shape
    delta = np.frombuffer(zlib.decompress(comp_bytes), dtype=np.uint8).reshape(H, W)
    return _delta_decode(delta)

def encode_colour_channel(channel, mode, quality_factor=1.0):
    if mode == 'LOSSLESS':
        data, shape = encode_channel_lossless(channel)
        return {'mode': 'LOSSLESS', 'data': data, 'shape': shape}
    else:
        comp, qt, shape = encode_channel_dct_compact(channel, quality_factor)
        return {'mode': 'LOSSY', 'comp': comp, 'quant': qt, 'shape': shape}

def decode_colour_channel(enc):
    if enc['mode'] == 'LOSSLESS':
        return decode_channel_lossless(enc['data'], enc['shape'])
    else:
        return decode_channel_dct_compact(enc['comp'], enc['quant'], enc['shape'])

print('Colour encoder v7 defined (compact binary DCT)')


Colour encoder v7 defined (compact binary DCT)


## A12 — Save / Load (.frac v7 binary format)

**Transform packing:** each transform is one 32-bit unsigned integer:
`[cy:6][cx:6][ti:3][cq:8][b:8]` (31 bits used, 1 spare).

- `cy`, `cx`: 6-bit domain block index (NOT pixel coordinate). At step=24, the
  maximum index is 63, covering images up to ~1528px per side.
- `ti`: 0–7, indexes the D4 transform applied to the domain block.
- `cq`: contrast quantised to uint8 via `cq + 127` (range [−0.75, 0.75] → stored).
- `b`: brightness stored as uint8 [0, 255] — v7 fix; prior versions used int8
  causing systematic underexposure for luminance > 127.

In [16]:
MAGIC, VERSION = b'FRAC', 7

def _pack_transform(cy, cx, ti, cq, bq):
    assert 0 <= cy <= 63, f'cy={cy} overflows 6-bit field'
    assert 0 <= cx <= 63, f'cx={cx} overflows 6-bit field'
    
    # ── High-Precision Quantisation (v7.3) ──
    # We scale the contractivity cq (range [-0.75, 0.75]) to fill the 
    # 8-bit integer field (0-255) centered at 127. 
    # This preserves float precision that was previously lost to truncation.
    cq_quant = int((cq / 0.75) * 127 + 127)
    
    packed = (
        (int(cy) & 0x3F) << 26 |
        (int(cx) & 0x3F) << 20 |
        (int(ti) & 0x07) << 17 |
        (int(cq_quant) & 0xFF) << 9 |
        (int(bq) & 0xFF) << 1
    )
    return struct.pack('>I', packed)

def _unpack_transform(data):
    p = struct.unpack('>I', data)[0]
    cy = (p >> 26) & 0x3F
    cx = (p >> 20) & 0x3F
    ti = (p >> 17) & 0x07
    
    # ── High-Precision Unpacking (v7.3) ──
    # Rescale the 8-bit integer back to floating point range [-0.75, 0.75].
    cq_packed = (p >> 9) & 0xFF
    cq = (float(cq_packed) - 127.0) * (0.75 / 127.0)
    
    bq = (p >> 1) & 0xFF
    return cy, cx, ti, cq, bq

def save_compressed(filepath, image_shape, orig_shape, y_transforms,
                    cb_enc, cr_enc, cfg, y2_transforms=None):
    H, W   = image_shape
    oH, oW = orig_shape
    ds     = cfg['domain_step']
    mode_b = 1 if cfg.get('colour_mode', 'LOSSY') == 'LOSSLESS' else 0
    has_r  = 1 if y2_transforms is not None else 0

    import pickle
    if cb_enc['mode'] == 'LOSSLESS':
        cb_blob = cb_enc['data']
        cr_blob = cr_enc['data']
    else:
        cb_blob = pickle.dumps({'comp': cb_enc['comp'],
                                'quant': cb_enc['quant'],
                                'shape': cb_enc['shape']})
        cr_blob = pickle.dumps({'comp': cr_enc['comp'],
                                'quant': cr_enc['quant'],
                                'shape': cr_enc['shape']})

    with open(filepath, 'wb') as f:
        f.write(MAGIC)
        f.write(struct.pack('I', VERSION))
        f.write(struct.pack('I', H))
        f.write(struct.pack('I', W))
        f.write(struct.pack('I', oH))
        f.write(struct.pack('I', oW))
        f.write(struct.pack('I', len(y_transforms)))
        f.write(struct.pack('I', ds))
        f.write(struct.pack('B', mode_b))
        f.write(struct.pack('B', has_r))
        f.write(struct.pack('I', len(cb_blob)))
        f.write(struct.pack('I', len(cr_blob)))
        for dy, dx, ti, cq, bq in y_transforms:
            cy_idx = dy // ds
            cx_idx = dx // ds
            f.write(_pack_transform(cy_idx, cx_idx, ti, cq, bq))
        if has_r:
            f.write(struct.pack('I', len(y2_transforms)))
            for dy, dx, ti, cq, bq in y2_transforms:
                cy_idx = dy // ds
                cx_idx = dx // ds
                f.write(_pack_transform(cy_idx, cx_idx, ti, cq, bq))
        f.write(cb_blob)
        f.write(cr_blob)

    sz = os.path.getsize(filepath)
    print(f"Saved '{filepath}'  ({sz/1024:.1f} KB)")


def load_compressed(filepath):
    import pickle
    with open(filepath, 'rb') as f:
        magic = f.read(4)
        assert magic == MAGIC, f'Bad magic: {magic}'
        version = struct.unpack('I', f.read(4))[0]
        H, W, oH, oW = struct.unpack('4I', f.read(16))
        n_y    = struct.unpack('I', f.read(4))[0]
        ds     = struct.unpack('I', f.read(4))[0]
        mode_b = struct.unpack('B', f.read(1))[0]
        has_r  = struct.unpack('B', f.read(1))[0]
        n_cb   = struct.unpack('I', f.read(4))[0]
        n_cr   = struct.unpack('I', f.read(4))[0]

        y_transforms = []
        for _ in range(n_y):
            cy, cx, ti, cq, bq = _unpack_transform(f.read(4))
            y_transforms.append((cy*ds, cx*ds, ti, cq, bq))

        y2_transforms = None
        if has_r:
            n_y2 = struct.unpack('I', f.read(4))[0]
            y2_transforms = []
            for _ in range(n_y2):
                cy, cx, ti, cq, bq = _unpack_transform(f.read(4))
                y2_transforms.append((cy*ds, cx*ds, ti, cq, bq))

        cb_blob = f.read(n_cb)
        cr_blob = f.read(n_cr)

    colour_mode = 'LOSSLESS' if mode_b else 'LOSSY'
    if colour_mode == 'LOSSLESS':
        cb_enc = {'mode':'LOSSLESS','data':cb_blob,'shape':(H,W)}
        cr_enc = {'mode':'LOSSLESS','data':cr_blob,'shape':(H,W)}
    else:
        cbd = pickle.loads(cb_blob)
        crd = pickle.loads(cr_blob)
        cb_enc = {'mode':'LOSSY','comp':cbd['comp'],'quant':cbd['quant'],'shape':cbd['shape']}
        cr_enc = {'mode':'LOSSY','comp':crd['comp'],'quant':crd['quant'],'shape':crd['shape']}

    cfg = {'domain_step': ds, 'colour_mode': colour_mode,
           'error_threshold': 0, 'flat_means': {}, 'flat_mask': [],
           'flat_pct': 0, 'median_var': 0, 'total_blocks': 0, 'range_vars': []}
    return (H, W), (oH, oW), y_transforms, cb_enc, cr_enc, cfg, y2_transforms

print('Save/Load defined (v7.3 High-Precision 8-bit Quantisation)')


Save/Load defined


## A13 — Utilities

Size reporting, side-by-side display, and quality metrics (PSNR, SSIM, MAE).

In [17]:
def print_size_report(img, y_tf, cb_enc, cr_enc, y2_tf=None, filepath=None):
    raw = img.nbytes
    yr  = len(y_tf) * 4
    y2r = len(y2_tf) * 4 if y2_tf else 0
    cbr = len(cb_enc.get('comp', cb_enc.get('data', b'')))
    crr = len(cr_enc.get('comp', cr_enc.get('data', b'')))
    tot = yr + y2r + cbr + crr
    r   = lambda a,b: f'{a/b:.3f}x (+{(1-a/b)*100:.1f}%)'
    print(f"{'='*55}")
    print(f'  Original           : {raw/1024:.1f} KB')
    print(f'  Y pass 1 (fractal) : {yr/1024:.1f} KB  {r(yr,raw)}')
    if y2_tf: print(f'  Y pass 2 (residual): {y2r/1024:.1f} KB')
    print(f'  Cb ({cb_enc["mode"]:9s})   : {cbr/1024:.1f} KB  {r(cbr,raw)}')
    print(f'  Cr ({cr_enc["mode"]:9s})   : {crr/1024:.1f} KB  {r(crr,raw)}')
    print(f"  {'─'*47}")
    print(f'  Total              : {tot/1024:.1f} KB  {r(tot,raw)}')
    if filepath and os.path.exists(filepath):
        print(f'  File on disk       : {os.path.getsize(filepath)/1024:.1f} KB')
    print(f"{'='*55}\n")

def side_by_side(a, b):
    H, W, _ = a.shape
    c = np.ones((H, W*2+4, 3), dtype=np.uint8) * 255
    c[:, :W, :] = a; c[:, W+4:, :] = b
    return Image.fromarray(c)

def compute_metrics(original, reconstructed):
    from scipy.ndimage import uniform_filter
    a = original.astype(np.float64)
    b = reconstructed.astype(np.float64)
    mse  = np.mean((a-b)**2)
    psnr = 10*np.log10(255**2/mse) if mse > 0 else float('inf')
    mae  = np.mean(np.abs(a-b))
    ch   = [np.mean((a[:,:,c]-b[:,:,c])**2) for c in range(3)]
    def ssim_c(x, y, L=255, k1=0.01, k2=0.03):
        w=11; mx=uniform_filter(x,w); my=uniform_filter(y,w)
        mxx=uniform_filter(x*x,w); myy=uniform_filter(y*y,w); mxy=uniform_filter(x*y,w)
        sx=mxx-mx**2; sy=myy-my**2; sxy=mxy-mx*my
        C1,C2=(k1*L)**2,(k2*L)**2
        return np.mean(((2*mx*my+C1)*(2*sxy+C2))/((mx**2+my**2+C1)*(sx+sy+C2)))
    ssim = np.mean([ssim_c(a[:,:,c], b[:,:,c]) for c in range(3)])
    print(f"\n{'='*45}")
    print(f'  PSNR : {psnr:.2f} dB', end='  ')
    if psnr>40: print('(excellent)')
    elif psnr>35: print('(good)')
    elif psnr>30: print('(acceptable)')
    elif psnr>25: print('(noticeable)')
    else: print('(significant loss)')
    print(f'  SSIM : {ssim:.4f}\n  MAE  : {mae:.2f}\n  MSE  : {mse:.2f}')
    print(f'  Chan : R={ch[0]:.1f} G={ch[1]:.1f} B={ch[2]:.1f}')
    if max(ch) / (min(ch) + 1e-9) > 2:
        print('  WARNING: channel imbalance — possible colour shift')
    print(f"{'='*45}\n")
    return {'mse': mse, 'psnr': psnr, 'ssim': ssim, 'mae': mae}

print('Utilities defined')


Utilities defined


## A14 — CNN Post-Decode Enhancement (Optional)

A lightweight residual CNN (~37K parameters) trained to recover high-frequency
content (edges, texture) lost by the fractal 2× downsampling lowpass filter.

The CNN learns *across-image* priors from diverse datasets, bypassing the
within-image self-similarity limitation that causes attractor mismatch on
high-entropy content.

**Requirements:** `torch` and the weights file `fractal_cnn_weights.pt`.
If weights are not found, the CNN step is skipped automatically.

**Effect on file size:** zero. The CNN is part of the decoder, not the
compressed file. `.frac` output is identical with or without CNN.

In [18]:
# ── CNN Post-Decode Enhancement Setup ──
# Set USE_CNN_ENHANCE = True to apply CNN after fractal decode.
# Requires: fractal_cnn_weights.pt in the same directory as this notebook.

USE_CNN_ENHANCE = True

_cnn_model = None

try:
    import torch
    import sys, os
    # Add notebook directory to path for fractal_cnn import
    _nb_dir = os.path.dirname(os.path.abspath('__file__'))
    if _nb_dir not in sys.path:
        sys.path.insert(0, _nb_dir)
    from fractal_cnn import load_model, cnn_enhance

    # Try to find weights file
    _weights_candidates = [
        'fractal_cnn_weights.pt',
        os.path.join(_nb_dir, 'fractal_cnn_weights.pt'),
        '/content/fractal_cnn_weights.pt',
    ]
    _weights_path = None
    for _wp in _weights_candidates:
        if os.path.exists(_wp):
            _weights_path = _wp
            break

    if _weights_path:
        _cnn_model = load_model(_weights_path)
        print(f'CNN enhancement: ENABLED (weights from {_weights_path})')
    else:
        print('CNN enhancement: weights not found \u2014 skipping CNN.')
        print('  Train with: !python fractal_cnn_train.py --datasets kodak,bsd500,div2k')
        USE_CNN_ENHANCE = False
except ImportError as e:
    print(f'CNN enhancement: DISABLED ({e})')
    print('  Install PyTorch: !pip install torch torchvision -q')
    USE_CNN_ENHANCE = False


CNN enhancement: DISABLED (No module named 'fractal_cnn')
  Install PyTorch: !pip install torch torchvision -q


## B — Main Pipeline

Encode or decode a single image. Set `IMAGE_PATH`, `OUTPUT_PATH`, and `ENCODE` in
Cell A3 before running.

Pipeline steps:
1. RGB → YCbCr
2. Image analysis (variance, threshold, step)
3. Pre-sample diagnostic
4. Fractal encode Y (batched GPU)
5. Residual pass (if gate conditions met)
6. Compact DCT encode Cb, Cr
7. Save .frac v7
8. Decode Y
9. CNN enhancement (if enabled)
10. Decode colour, reconstruct RGB, display
11. Quality metrics (with CNN improvement comparison)

In [ ]:
from PIL import Image
import numpy as np
from IPython.display import display

if ENCODE:
    img_raw = np.array(Image.open(IMAGE_PATH).convert('RGB'))
    print(f'Image: {img_raw.shape[1]}x{img_raw.shape[0]}  '\
          f'{img_raw.nbytes/1024:.1f} KB raw')
    img, orig_H, orig_W = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]

    print('\n[1] RGB -> YCbCr')
    ycbcr = rgb_to_ycbcr(img)

    print('\n[2] Image analysis...')
    cfg = analyse_image(ycbcr[:,:,0], TIME_BUDGET_SECONDS, AUTO_THRESHOLD)
    if not AUTO_THRESHOLD:
        cfg['domain_step']     = DOMAIN_STEP
        cfg['error_threshold'] = ERROR_THRESHOLD
    cfg['colour_mode'] = COMPRESSION_MODE

    print('\n[3] Pre-sample diagnostic...')
    print('    Building domain stack...')
    sds, smeta = build_domain_stack(ycbcr[:,:,0].astype(np.float32), cfg['domain_step'])
    cfg = run_presample_diagnostic(ycbcr[:,:,0], sds, cfg, 0.05)

    print(f'\n[4] Fractal encode Y  '\
          f'(thresh={cfg["error_threshold"]:.1f}  step={cfg["domain_step"]})')
    y_tf, actual_early_pct = encode_channel_gpu(ycbcr[:,:,0], cfg)

    y2_tf = None
    if ENCODE_RESIDUAL:
        if RESIDUAL_MIN_EARLY < actual_early_pct < RESIDUAL_MAX_EARLY:
            print(f'\n[4b] Residual Y encoding '\
                  f'(early_pct={actual_early_pct*100:.1f}% in gate '\
                  f'[{RESIDUAL_MIN_EARLY*100:.0f}%-{RESIDUAL_MAX_EARLY*100:.0f}%])...')
            y_rec_for_res = decode_fractal(y_tf, (H_pad, W_pad), n_iterations=N_ITERATIONS)
            y2_tf = encode_residual(ycbcr[:,:,0], y_rec_for_res, cfg)
        else:
            reason = ('too high — error is random noise'\
                      if actual_early_pct >= RESIDUAL_MAX_EARLY\
                      else 'too low — error is diffuse gradient')
            print(f'\n[4b] Residual skipped '\
                  f'(early_pct={actual_early_pct*100:.1f}% {reason})')

    print(f'\n[5] Encode colour channels  (mode={COMPRESSION_MODE})')
    cb_enc = encode_colour_channel(ycbcr[:,:,1], COMPRESSION_MODE, DCT_QUALITY_FACTOR)
    cr_enc = encode_colour_channel(ycbcr[:,:,2], COMPRESSION_MODE, DCT_QUALITY_FACTOR)

    print_size_report(img, y_tf, cb_enc, cr_enc, y2_tf)
    print(f'\n[6] Saving to {OUTPUT_PATH}')
    save_compressed(OUTPUT_PATH, img.shape[:2], (orig_H, orig_W),\
                    y_tf, cb_enc, cr_enc, cfg, y2_tf)
    img_display = img_raw[:orig_H, :orig_W]

else:
    print(f'\n[0] Loading from {OUTPUT_PATH}')
    padded_shape, (orig_H, orig_W), y_tf, cb_enc, cr_enc, cfg, y2_tf = load_compressed(OUTPUT_PATH)
    H_pad, W_pad = padded_shape
    # Attempt to load original for metrics if path exists, else dummy
    try:
        img_display = np.array(Image.open(IMAGE_PATH).convert('RGB'))[:orig_H, :orig_W]
    except:
        img_display = None

print(f'\n[7] Decode Y ({N_ITERATIONS} iterations)...')
if y2_tf is not None:
    y_rec = decode_with_residual(y_tf, y2_tf, (H_pad, W_pad), N_ITERATIONS)
    y_rec_base = None # Residual bypasses CNN usually
else:
    y_rec = decode_fractal(y_tf, (H_pad, W_pad), N_ITERATIONS)
    y_rec_base = y_rec.copy()
    if USE_CNN_ENHANCE and _cnn_model is not None:
        print('\n[7b] CNN post-decode enhancement...')
        from fractal_cnn import cnn_enhance
        y_rec = cnn_enhance(y_rec, _cnn_model)

print('\n[8] Decode colour channels')
cb_rec = decode_colour_channel(cb_enc)
cr_rec = decode_colour_channel(cr_enc)

print('\n[9] YCbCr -> RGB')
result_padded = ycbcr_to_rgb(np.stack([y_rec, cb_rec, cr_rec], axis=2))
result        = crop_to_original(result_padded, orig_H, orig_W)

if img_display is not None:
    display(side_by_side(img_display, result))
    print('\n[10] Quality metrics')
    metrics = compute_metrics(img_display, result)
else:
    display(Image.fromarray(result))
    print('\n[10] Quality metrics skipped (original image not loaded)')

## C1 — Upscaling Comparison (Round-Trip Evaluation)

Compares fractal resolution-independent upscaling against bicubic and Lanczos
interpolation at 2×, 4×, and 8×.

**Round-trip methodology:** each method produces a scale× output; that output is
area-downscaled back to the original resolution using pixel averaging; PSNR/SSIM
is measured against the original. Area averaging introduces no algorithm signature,
ensuring no method is evaluated against its own output as a reference.

All methods are compared at equal storage budget matched to the fractal file size.

Set `UPSCALE_TARGETS` below with the original image path and corresponding .frac file.

In [ ]:
import cv2
from datetime import datetime
try:
    from skimage.metrics import structural_similarity as ssim_fn
except ImportError:
    !pip install scikit-image -q
    from skimage.metrics import structural_similarity as ssim_fn

UPSCALE_TARGETS = [
    {
        "label":   "Portrait",
        "orig":    "/content/your_image.png",   # ← set original image path
        "frac":    "/content/output.frac",       # ← set .frac file path
        "frac_kb": 102.5,                        # ← set from Cell B output
    },
    # Add more entries for additional images
]
SCALES   = [2, 4, 8]
LOG_PATH = "/content/upscale_comparison_log.json"

def img_metrics_up(orig_rgb, recon_rgb):
    o = orig_rgb.astype(np.float64); r = recon_rgb.astype(np.float64)
    mse = np.mean((o - r) ** 2)
    if mse == 0: return float('inf'), 1.0
    psnr = 10 * np.log10(255.0**2 / mse)
    og = cv2.cvtColor(orig_rgb,  cv2.COLOR_RGB2GRAY)
    rg = cv2.cvtColor(recon_rgb, cv2.COLOR_RGB2GRAY)
    s  = ssim_fn(og, rg, data_range=255)
    return round(psnr, 2), round(float(s), 3)

def area_downscale(img_rgb, scale):
    H, W = img_rgb.shape[:2]
    return cv2.resize(img_rgb, (W//scale, H//scale), interpolation=cv2.INTER_AREA)

def find_jpeg_quality(img_rgb, target_kb):
    best_q, best_diff = 1, float('inf')
    for q in range(1, 96, 2):
        bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        diff = abs(len(buf)/1024.0 - target_kb)
        if diff < best_diff: best_diff, best_q = diff, q
    return best_q

def upscale_baseline(orig_rgb, scale, target_kb, interp_flag):
    H, W   = orig_rgb.shape[:2]
    small  = area_downscale(orig_rgb, scale)
    best_q = find_jpeg_quality(small, target_kb)
    bgr    = cv2.cvtColor(small, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, best_q])
    small_dec = cv2.cvtColor(cv2.imdecode(np.frombuffer(buf, np.uint8),
                             cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    kb = len(buf)/1024
    upscaled  = cv2.resize(small_dec, (W, H), interpolation=interp_flag)
    roundtrip = cv2.resize(upscaled,  (W//scale, H//scale), interpolation=cv2.INTER_AREA)
    roundtrip = cv2.resize(roundtrip, (W, H), interpolation=cv2.INTER_AREA)
    return roundtrip, round(kb,1), best_q

all_results = {}
for tgt in UPSCALE_TARGETS:
    label, orig_path, frac_path, frac_kb =         tgt['label'], tgt['orig'], tgt['frac'], tgt['frac_kb']
    if not os.path.exists(orig_path) or not os.path.exists(frac_path):
        print(f'[SKIP] {label} — file not found'); continue

    orig_rgb = np.array(Image.open(orig_path).convert('RGB'))
    H, W     = orig_rgb.shape[:2]
    padded_shape, (orig_H, orig_W), y_tf, cb_enc, cr_enc, fcfg, y2_tf =         load_compressed(frac_path)

    image_results = {}
    print(f'\n══ {label} ({W}×{H}) ══')
    print(f'  {"Scale":>6}  {"Fractal":>8}  {"Bicubic":>8}  {"Lanczos":>8}  {"Δ PSNR":>8}')

    for scale in SCALES:
        frac_up = decode_and_upscale(y_tf, cb_enc, cr_enc,
                                      padded_shape, (orig_H, orig_W),
                                      scale=scale, n_i=N_ITERATIONS+2)[:H*scale, :W*scale]
        frac_rt = cv2.resize(frac_up, (W, H), interpolation=cv2.INTER_AREA)
        f_psnr, f_ssim = img_metrics_up(orig_rgb, frac_rt)

        bic_rt, bic_kb, bic_q = upscale_baseline(orig_rgb, scale, frac_kb, cv2.INTER_CUBIC)
        b_psnr, b_ssim = img_metrics_up(orig_rgb, bic_rt)

        lz_rt, lz_kb, lz_q = upscale_baseline(orig_rgb, scale, frac_kb, cv2.INTER_LANCZOS4)
        l_psnr, l_ssim = img_metrics_up(orig_rgb, lz_rt)

        best = max(b_psnr, l_psnr)
        delta = round(f_psnr - best, 2)
        verdict = '◀ fractal' if delta > 0 else '  baseline'
        print(f'  {scale}×  →  {f_psnr:>8.2f}  {b_psnr:>8.2f}  {l_psnr:>8.2f}  '
              f'{delta:>+8.2f}  {verdict}')

        image_results[f'{scale}x'] = {
            'fractal': {'psnr':f_psnr,'ssim':f_ssim,'kb':frac_kb},
            'bicubic': {'psnr':b_psnr,'ssim':b_ssim,'kb':bic_kb,'jpeg_q':bic_q},
            'lanczos': {'psnr':l_psnr,'ssim':l_ssim,'kb':lz_kb, 'jpeg_q':lz_q},
            'delta_psnr_vs_baseline': delta,
        }
    all_results[label] = image_results

with open(LOG_PATH, 'w') as f: json.dump(all_results, f, indent=2)
print(f'\nSaved → {LOG_PATH}')


## C2 — JPEG Rate-Distortion Comparison

For each benchmark image, finds the JPEG quality whose file size matches the
fractal file size, then measures PSNR/SSIM. Also measures PNG lossless size.

Set `BENCHMARK_IMAGES` with per-image metadata from the main pipeline output.

In [ ]:
from datetime import datetime
try:
    import cv2
except: !pip install opencv-python-headless -q; import cv2

# ── SET THESE from your Cell B outputs ──────────────────────────────
BENCHMARK_IMAGES = [
    {"label":"Portrait",          "orig":"/content/portrait.png",  "frac_kb":102.5, "frac_psnr":37.03, "frac_ssim":0.963},
    {"label":"Architectural",     "orig":"/content/arch.png",      "frac_kb":12.8,  "frac_psnr":40.57, "frac_ssim":0.990},
    {"label":"Aerial footage",    "orig":"/content/aerial.png",    "frac_kb":112.6, "frac_psnr":26.02, "frac_ssim":0.880},
    {"label":"Brick wall",        "orig":"/content/brick.png",     "frac_kb":239.9, "frac_psnr":25.48, "frac_ssim":0.672},
    {"label":"Forest canopy",     "orig":"/content/canopy.png",    "frac_kb":172.1, "frac_psnr":20.70, "frac_ssim":0.727},
]

def img_metrics_jpeg(orig, recon):
    a=orig.astype(np.float64); b=recon.astype(np.float64)
    mse=np.mean((a-b)**2)
    return round(10*np.log10(255**2/mse) if mse>0 else 100.0, 2)

def find_nearest_jpeg(img_rgb, target_kb):
    best_q, best_diff = 1, float('inf')
    for q in range(1, 96):
        buf = io.BytesIO()
        Image.fromarray(img_rgb).save(buf, 'JPEG', quality=q)
        diff = abs(buf.tell()/1024 - target_kb)
        if diff < best_diff: best_diff, best_q = diff, q
    return best_q

results = {}
print(f'{"Image":<22} {"Frac KB":>8} {"Frac PSNR":>10} {"JPEG q":>7} {"JPEG KB":>8} {"JPEG PSNR":>10} {"PNG KB":>8}')
print('─'*75)
for cfg in BENCHMARK_IMAGES:
    if not os.path.exists(cfg['orig']): print(f'[SKIP] {cfg["label"]}'); continue
    img_rgb  = np.array(Image.open(cfg['orig']).convert('RGB'))
    q        = find_nearest_jpeg(img_rgb, cfg['frac_kb'])
    buf = io.BytesIO(); Image.fromarray(img_rgb).save(buf,'JPEG',quality=q)
    kb  = round(buf.tell()/1024,1); buf.seek(0)
    dec = np.array(Image.open(buf).convert('RGB'))
    j_psnr = img_metrics_jpeg(img_rgb, dec)
    png_buf = io.BytesIO(); Image.fromarray(img_rgb).save(png_buf,'PNG')
    png_kb  = round(png_buf.tell()/1024,1)
    results[cfg['label']] = {**cfg,'jpeg_quality':q,'jpeg_kb':kb,'jpeg_psnr':j_psnr,'png_kb':png_kb}
    print(f'{cfg["label"]:<22} {cfg["frac_kb"]:>8.1f} {cfg["frac_psnr"]:>10.2f} '
          f'{q:>7} {kb:>8.1f} {j_psnr:>10.2f} {png_kb:>8.1f}')

with open('/content/jpeg_comparison_log.json','w') as f: json.dump(results,f,indent=2)
print('\nSaved → /content/jpeg_comparison_log.json')


## C3 — Rate-Distortion Benchmark

Sweeps fractal error threshold to produce a rate-distortion curve and compares against JPEG. Uncomment the last line to run.

In [ ]:
def run_rate_distortion(image_path, thresholds=None, n_iter=10):
    from scipy.ndimage import uniform_filter
    if thresholds is None: thresholds = [5, 15, 30, 50, 80, 120, 200]

    img_raw = np.array(Image.open(image_path).convert('RGB'))
    img, oH, oW = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]
    ycbcr = rgb_to_ycbcr(img)
    orig  = img_raw

    base_cfg = analyse_image(ycbcr[:,:,0], TIME_BUDGET_SECONDS, False)
    base_cfg['domain_step']  = DOMAIN_STEP
    base_cfg['colour_mode']  = 'LOSSY'
    ds, _ = build_domain_stack(ycbcr[:,:,0].astype(np.float32), DOMAIN_STEP)

    def ssim_c(x, y, L=255, k1=0.01, k2=0.03):
        w=11; mx=uniform_filter(x,w); my=uniform_filter(y,w)
        mxx=uniform_filter(x*x,w); myy=uniform_filter(y*y,w); mxy=uniform_filter(x*y,w)
        sx=mxx-mx**2; sy=myy-my**2; sxy=mxy-mx*my
        C1,C2=(k1*L)**2,(k2*L)**2
        return np.mean(((2*mx*my+C1)*(2*sxy+C2))/((mx**2+my**2+C1)*(sx+sy+C2)))

    fpts = []
    for thresh in thresholds:
        rc = dict(base_cfg); rc['error_threshold'] = thresh
        ytf, _ = encode_channel_gpu(ycbcr[:,:,0], rc)
        cbe = encode_colour_channel(ycbcr[:,:,1], 'LOSSY', DCT_QUALITY_FACTOR)
        cre = encode_colour_channel(ycbcr[:,:,2], 'LOSSY', DCT_QUALITY_FACTOR)
        tmp = f'/tmp/rd_{thresh}.frac'
        save_compressed(tmp, img.shape[:2], (oH,oW), ytf, cbe, cre, rc)
        sz  = os.path.getsize(tmp) / 1024
        yr  = decode_fractal(ytf, (H_pad,W_pad), n_iter)
        res = crop_to_original(ycbcr_to_rgb(np.stack(
            [yr, decode_colour_channel(cbe), decode_colour_channel(cre)], axis=2)), oH, oW)
        a=orig.astype(np.float64); b=res.astype(np.float64)
        mse=np.mean((a-b)**2); psnr=10*np.log10(255**2/mse)
        ssim=np.mean([ssim_c(a[:,:,c],b[:,:,c]) for c in range(3)])
        fpts.append((sz, psnr, ssim, thresh))
        print(f'  T={thresh:3d}  {sz:.0f}KB  PSNR={psnr:.2f}  SSIM={ssim:.4f}')

    jpts = []
    for q in [5,10,20,30,50,70,85,95]:
        buf=io.BytesIO(); Image.fromarray(orig).save(buf,format='JPEG',quality=q)
        sz=buf.tell()/1024; buf.seek(0); j=np.array(Image.open(buf))
        a=orig.astype(np.float64); b=j.astype(np.float64)
        mse=np.mean((a-b)**2); psnr=10*np.log10(255**2/mse)
        ssim=np.mean([ssim_c(a[:,:,c],b[:,:,c]) for c in range(3)])
        jpts.append((sz, psnr, ssim, q))

    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
    ax1.plot([p[0] for p in jpts],[p[1] for p in jpts],'b-o',label='JPEG',lw=2)
    ax1.plot([p[0] for p in fpts],[p[1] for p in fpts],'r-s',label='Fractal v7',lw=2)
    ax1.set_xlabel('File size (KB)'); ax1.set_ylabel('PSNR (dB)')
    ax1.set_title('Rate-Distortion'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot([p[0] for p in jpts],[p[2] for p in jpts],'b-o',label='JPEG',lw=2)
    ax2.plot([p[0] for p in fpts],[p[2] for p in fpts],'r-s',label='Fractal v7',lw=2)
    ax2.set_xlabel('File size (KB)'); ax2.set_ylabel('SSIM')
    ax2.set_title('Rate-Distortion (SSIM)'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/rate_distortion_v7.png', dpi=150, bbox_inches='tight')
    plt.show()
    return fpts, jpts

print('Rate-distortion benchmark defined')
# fpts, jpts = run_rate_distortion(IMAGE_PATH)   # ← uncomment to run


## D1 — Dataset Evaluation (Kodak-24 + BSD100)

Runs the full pipeline across a standard benchmark dataset to validate results
beyond the 5-image benchmark set.

**Why Kodak:** 24 images, 768×512, standardised across the compression literature.
Available without authentication from a stable academic mirror.

**Output:** `results_kodak.json` — saved incrementally so a crash does not lose
progress. Each entry includes all metrics needed to update the paper tables.

Estimated runtime: ~25 minutes for all 24 Kodak images on T4.

BSD100 (50 images) is included but commented out — run in a second session.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL D — DATASET EVALUATION (Kodak 24 + BSD100 subset)
# ║
# ║  PURPOSE: produce per-image results across a standard benchmark dataset
# ║           to back the 0.50 threshold multiplier and 40-70% residual gate
# ║           with data rather than assertion.
# ║
# ║  OUTPUT:  results_kodak.json and results_bsd.json
# ║           ready to import into the paper's expanded Table III.
# ║
# ║  RUNTIME: ~1 Colab session (Kodak 24 images ≈ 20-30 min on T4)
# ║           BSD100 subset adds another ~30 min.
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, io, time, requests, zipfile
import numpy as np
from PIL import Image

# ── dataset download helpers ──────────────────────────────────────────────

def download_kodak(dest='/content/kodak'):
    """
    Download all 24 Kodak images (768x512 or 512x768, lossless PNG mirrors).
    Source: r0k.us/graphics/kodak — standard academic mirror, no auth required.
    """
    os.makedirs(dest, exist_ok=True)
    base = 'http://r0k.us/graphics/kodak/kodak/'
    downloaded = []
    for i in range(1, 25):
        fname = f'kodim{i:02d}.png'
        fpath = os.path.join(dest, fname)
        if os.path.exists(fpath):
            downloaded.append(fpath)
            continue
        url = base + fname
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(fpath, 'wb') as f:
                    f.write(r.content)
                downloaded.append(fpath)
                print(f'  Downloaded {fname}')
            else:
                print(f'  SKIP {fname} — HTTP {r.status_code}')
        except Exception as e:
            print(f'  FAIL {fname}: {e}')
    print(f'Kodak: {len(downloaded)}/24 images available')
    return sorted(downloaded)


def download_bsd100_subset(dest='/content/bsd100', n=50):
    """
    Download BSD100 subset via BSDS500 archive (Berkeley segmentation dataset).
    Uses the standard academic download from eecs.berkeley.edu.
    Falls back to a public mirror if primary fails.
    Returns up to n image paths.
    """
    os.makedirs(dest, exist_ok=True)
    existing = sorted([
        os.path.join(dest, f) for f in os.listdir(dest)
        if f.endswith('.jpg') or f.endswith('.png')
    ])
    if len(existing) >= n:
        print(f'BSD: {len(existing)} images already present, using first {n}')
        return existing[:n]

    # Primary source: BSDS500 zip from Berkeley
    url = 'https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz'
    print(f'Downloading BSD500 archive (may take ~1 min)...')
    try:
        r = requests.get(url, timeout=120, stream=True)
        if r.status_code == 200:
            import tarfile, io as _io
            buf = _io.BytesIO(r.content)
            with tarfile.open(fileobj=buf, mode='r:gz') as tar:
                imgs = [m for m in tar.getmembers()
                        if '/images/test/' in m.name and m.name.endswith('.jpg')][:n]
                for m in imgs:
                    fname = os.path.basename(m.name)
                    fpath = os.path.join(dest, fname)
                    f = tar.extractfile(m)
                    if f:
                        with open(fpath, 'wb') as out:
                            out.write(f.read())
            print(f'  Extracted {len(imgs)} BSD test images')
            return sorted([os.path.join(dest, os.path.basename(m.name)) for m in imgs])
        else:
            print(f'  BSD download failed: HTTP {r.status_code}')
            return []
    except Exception as e:
        print(f'  BSD download error: {e}')
        return []


# ── single-image pipeline runner ─────────────────────────────────────────

def run_pipeline_on_image(image_path, time_budget=90):
    """
    Run the full v7_2 pipeline on one image.
    Returns a result dict with all metrics, or None on failure.
    """
    try:
        img_raw = np.array(Image.open(image_path).convert('RGB'))
        img, orig_H, orig_W = pad_to_block_multiple(img_raw)
        H_pad, W_pad = img.shape[:2]

        t_start = time.time()

        ycbcr = rgb_to_ycbcr(img)

        cfg = analyse_image(ycbcr[:,:,0], time_budget, auto_threshold=True)
        cfg['colour_mode'] = 'LOSSY'

        sds, smeta = build_domain_stack(
            ycbcr[:,:,0].astype(np.float32), cfg['domain_step'])
        cfg = run_presample_diagnostic(ycbcr[:,:,0], sds, cfg, 0.05)
        cfg['colour_mode'] = 'LOSSY'
        presample_rate = cfg.get('early_stop_pct', None)

        y_tf, actual_early_pct = encode_channel_gpu(ycbcr[:,:,0], cfg)

        y2_tf = None
        if ENCODE_RESIDUAL and RESIDUAL_MIN_EARLY < actual_early_pct < RESIDUAL_MAX_EARLY:
            y_rec = decode_fractal(y_tf, (H_pad, W_pad), N_ITERATIONS)
            y2_tf = encode_residual(ycbcr[:,:,0], y_rec, cfg)

        cb_enc = encode_colour_channel(ycbcr[:,:,1], 'LOSSY', DCT_QUALITY_FACTOR)
        cr_enc = encode_colour_channel(ycbcr[:,:,2], 'LOSSY', DCT_QUALITY_FACTOR)

        encode_time = time.time() - t_start

        # ── decode and measure quality ────────────────────────────────────
        if y2_tf is not None:
            y_dec = decode_with_residual(y_tf, y2_tf, (H_pad, W_pad), N_ITERATIONS)
        else:
            y_dec = decode_fractal(y_tf, (H_pad, W_pad), N_ITERATIONS)

        cb_dec = decode_colour_channel(cb_enc)
        cr_dec = decode_colour_channel(cr_enc)
        ycbcr_dec = np.stack([y_dec, cb_dec, cr_dec], axis=2)
        rgb_dec = np.clip(ycbcr_to_rgb(ycbcr_dec), 0, 255).astype(np.uint8)
        rgb_dec = rgb_dec[:orig_H, :orig_W]

        # file size estimate
        tf_bytes = len(y_tf) * 4
        if y2_tf: tf_bytes += len(y2_tf) * 4
        cb_bytes = len(cb_enc.get('comp', b''))
        cr_bytes = len(cr_enc.get('comp', b''))
        total_kb = (tf_bytes + cb_bytes + cr_bytes) / 1024

        # quality metrics
        orig_crop = img_raw[:orig_H, :orig_W].astype(np.float32)
        dec_f     = rgb_dec.astype(np.float32)
        mse  = float(np.mean((orig_crop - dec_f) ** 2))
        psnr = float(10 * np.log10(255**2 / mse)) if mse > 0 else 100.0
        ssim = float(_ssim(orig_crop, dec_f))

        raw_kb = img_raw.nbytes / 1024
        ratio  = 1.0 - total_kb / raw_kb

        return {
            'image':          os.path.basename(image_path),
            'resolution':     f'{orig_W}×{orig_H}',
            'raw_kb':         round(raw_kb, 1),
            'frac_kb':        round(total_kb, 1),
            'compression':    round(ratio * 100, 1),
            'psnr':           round(psnr, 2),
            'ssim':           round(ssim, 4),
            'encode_time_s':  round(encode_time, 1),
            'domain_step':    cfg['domain_step'],
            'candidates':     sds.shape[0],
            'presample_rate': round(presample_rate * 100, 1) if presample_rate else None,
            'early_stop_pct': round(actual_early_pct * 100, 1),
            'residual_used':  y2_tf is not None,
            'median_var':     round(cfg.get('median_var', 0), 1),
            'threshold':      round(cfg['error_threshold'], 1),
        }

    except Exception as e:
        print(f'  ERROR on {os.path.basename(image_path)}: {e}')
        return None


def _ssim(a, b, window=7):
    """Fast SSIM approximation (single-scale, luminance channel)."""
    from scipy.ndimage import uniform_filter
    a = a.mean(axis=2) if a.ndim == 3 else a
    b = b.mean(axis=2) if b.ndim == 3 else b
    k1, k2, L = 0.01, 0.03, 255.0
    C1, C2 = (k1*L)**2, (k2*L)**2
    mu1 = uniform_filter(a, window)
    mu2 = uniform_filter(b, window)
    mu1_sq, mu2_sq, mu1mu2 = mu1**2, mu2**2, mu1*mu2
    sig1_sq = uniform_filter(a*a, window) - mu1_sq
    sig2_sq = uniform_filter(b*b, window) - mu2_sq
    sig12   = uniform_filter(a*b, window) - mu1mu2
    num = (2*mu1mu2 + C1) * (2*sig12 + C2)
    den = (mu1_sq + mu2_sq + C1) * (sig1_sq + sig2_sq + C2)
    return float(np.mean(num / den))


# ── JPEG comparison helper ────────────────────────────────────────────────

def jpeg_at_matched_size(img_array, target_kb, tolerance_pct=5):
    """Binary search for JPEG quality matching target_kb."""
    lo, hi = 1, 95
    best_q, best_kb = lo, 0
    for _ in range(12):
        q = (lo + hi) // 2
        buf = io.BytesIO()
        Image.fromarray(img_array).save(buf, 'JPEG', quality=q)
        kb = buf.tell() / 1024
        if kb <= target_kb:
            best_q, best_kb = q, kb
            lo = q + 1
        else:
            hi = q - 1
    # measure quality at best_q
    buf = io.BytesIO()
    Image.fromarray(img_array).save(buf, 'JPEG', quality=best_q)
    buf.seek(0)
    jpeg_dec = np.array(Image.open(buf).convert('RGB')).astype(np.float32)
    orig_f   = img_array.astype(np.float32)
    mse      = float(np.mean((orig_f - jpeg_dec)**2))
    psnr     = float(10 * np.log10(255**2 / mse)) if mse > 0 else 100.0
    return {'quality': best_q, 'kb': round(best_kb, 1), 'psnr': round(psnr, 2)}


# ── main evaluation loop ──────────────────────────────────────────────────

def run_dataset_evaluation(image_paths, output_json, dataset_name='dataset',
                            time_budget=90, run_jpeg=True):
    results = []
    n = len(image_paths)
    print(f'\n{"="*60}')
    print(f'  {dataset_name}: evaluating {n} images')
    print(f'{"="*60}')

    for i, path in enumerate(image_paths):
        print(f'\n[{i+1}/{n}] {os.path.basename(path)}')
        r = run_pipeline_on_image(path, time_budget)
        if r is None:
            continue

        if run_jpeg:
            img_arr = np.array(Image.open(path).convert('RGB'))
            r['jpeg'] = jpeg_at_matched_size(img_arr, r['frac_kb'])

        results.append(r)
        print(f'  PSNR={r["psnr"]} dB  SSIM={r["ssim"]}  '
              f'{r["frac_kb"]} KB ({r["compression"]}%)  '
              f'early={r["early_stop_pct"]}%  '
              f'{r["encode_time_s"]}s')

        # save incrementally so a crash doesn't lose everything
        with open(output_json, 'w') as f:
            json.dump(results, f, indent=2)

    # ── summary statistics ────────────────────────────────────────────────
    if results:
        psnrs       = [r['psnr'] for r in results]
        ssims       = [r['ssim'] for r in results]
        ratios      = [r['compression'] for r in results]
        early_rates = [r['early_stop_pct'] for r in results]
        thresholds  = [r['threshold'] for r in results]

        # classify by early-stop rate into same content classes
        smooth  = [r for r in results if r['early_stop_pct'] >= 70]
        mixed   = [r for r in results if 40 <= r['early_stop_pct'] < 70]
        hard    = [r for r in results if r['early_stop_pct'] < 40]

        print(f'\n{"="*60}')
        print(f'  {dataset_name} SUMMARY ({len(results)} images)')
        print(f'{"="*60}')
        print(f'  Mean PSNR:         {np.mean(psnrs):.2f} dB  '
              f'(σ={np.std(psnrs):.2f})')
        print(f'  Mean SSIM:         {np.mean(ssims):.4f}')
        print(f'  Mean compression:  {np.mean(ratios):.1f}%')
        print(f'  Mean early-stop:   {np.mean(early_rates):.1f}%')
        print(f'  Mean threshold:    {np.mean(thresholds):.1f}')
        print(f'  Smooth (≥70%):     {len(smooth)} images  '
              f'mean PSNR={np.mean([r["psnr"] for r in smooth]):.2f} dB' if smooth else '')
        print(f'  Mixed (40-70%):    {len(mixed)} images  '
              f'mean PSNR={np.mean([r["psnr"] for r in mixed]):.2f} dB' if mixed else '')
        print(f'  Hard (<40%):       {len(hard)} images  '
              f'mean PSNR={np.mean([r["psnr"] for r in hard]):.2f} dB' if hard else '')
        print(f'\n  Results saved to {output_json}')

    return results


# ── ENTRY POINT — run this block ──────────────────────────────────────────

# 1. Kodak (24 images — primary benchmark)
kodak_paths = download_kodak('/content/kodak')
kodak_results = run_dataset_evaluation(
    kodak_paths,
    output_json='/content/results_kodak.json',
    dataset_name='Kodak-24',
    time_budget=90,
    run_jpeg=True
)

# 2. BSD100 subset (50 images — secondary validation)
# Uncomment for second session or if time permits:
# bsd_paths = download_bsd100_subset('/content/bsd100', n=50)
# bsd_results = run_dataset_evaluation(
#     bsd_paths,
#     output_json='/content/results_bsd.json',
#     dataset_name='BSD100-50',
#     time_budget=60,
#     run_jpeg=True
# )

print('\nDone. Download results_kodak.json from the Files panel.')


## D2 — Threshold Multiplier Sweep

Sweeps k ∈ {0.1 … 0.8} in τ = k × median_var across three representative images
to characterise the relationship between threshold and early-stop rate.

**Key finding from sweep data:** PSNR is invariant to k across all tested values.
The encoder always stores the globally optimal domain match (argmin over all N
candidates regardless of threshold); τ only classifies whether the optimal match
fell below the threshold for the early-stop rate counter. The 0.50 value is
justified by keeping the early-stop rate in the 51–70% range across diverse image
content, ensuring the pre-sample diagnostic and residual gate operate meaningfully.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL E — THRESHOLD MULTIPLIER SWEEP
# ║
# ║  PURPOSE: justify the 0.50 multiplier in τ = k × median_var
# ║           by sweeping k ∈ {0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8}
# ║           across a representative subset of Kodak images and measuring
# ║           PSNR and encode time at each k.
# ║
# ║  EXPECTED RESULT: 0.50 sits at the knee of the PSNR/time tradeoff curve.
# ║  OUTPUT: results_threshold_sweep.json
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, time
import numpy as np
from PIL import Image

# Use a representative subset: 1 smooth, 1 mixed, 1 texture image from Kodak
# kodim23 = smooth portrait-style, kodim05 = outdoor mixed, kodim15 = foliage
SWEEP_IMAGES = [
    '/content/kodak/kodim23.png',   # smooth
    '/content/kodak/kodim05.png',   # mixed
    '/content/kodak/kodim15.png',   # high entropy
]

MULTIPLIERS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

def encode_with_multiplier(image_path, k, time_budget=90):
    """
    Run full pipeline with threshold = k × median_var.
    Returns dict with psnr, ssim, encode_time_s, early_stop_pct.
    """
    img_raw = np.array(Image.open(image_path).convert('RGB'))
    img, orig_H, orig_W = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]
    ycbcr = rgb_to_ycbcr(img)

    # Run analysis but override threshold with k × median_var
    cfg = analyse_image(ycbcr[:,:,0], time_budget, auto_threshold=True)
    median_var = cfg['median_var']
    cfg['error_threshold'] = float(np.clip(median_var * k, 5.0, 500.0))
    cfg['colour_mode'] = 'LOSSY'

    sds, smeta = build_domain_stack(ycbcr[:,:,0].astype(np.float32), cfg['domain_step'])

    t0 = time.time()
    y_tf, actual_early_pct = encode_channel_gpu(ycbcr[:,:,0], cfg)
    encode_time = time.time() - t0

    # Decode and measure — no residual pass (isolate threshold effect)
    y_dec = decode_fractal(y_tf, (H_pad, W_pad), N_ITERATIONS)
    cb_enc = encode_colour_channel(ycbcr[:,:,1], 'LOSSY', DCT_QUALITY_FACTOR)
    cr_enc = encode_colour_channel(ycbcr[:,:,2], 'LOSSY', DCT_QUALITY_FACTOR)
    cb_dec = decode_colour_channel(cb_enc)
    cr_dec = decode_colour_channel(cr_enc)
    ycbcr_dec = np.stack([y_dec, cb_dec, cr_dec], axis=2)
    rgb_dec = np.clip(ycbcr_to_rgb(ycbcr_dec), 0, 255).astype(np.uint8)[:orig_H, :orig_W]

    orig_f = img_raw[:orig_H, :orig_W].astype(np.float32)
    dec_f  = rgb_dec.astype(np.float32)
    mse    = float(np.mean((orig_f - dec_f)**2))
    psnr   = float(10 * np.log10(255**2 / mse)) if mse > 0 else 100.0
    ssim   = float(_ssim(orig_f, dec_f))

    return {
        'k':              k,
        'threshold':      round(cfg['error_threshold'], 1),
        'median_var':     round(median_var, 1),
        'early_stop_pct': round(actual_early_pct * 100, 1),
        'psnr':           round(psnr, 2),
        'ssim':           round(ssim, 4),
        'encode_time_s':  round(encode_time, 1),
    }


sweep_results = {}

for img_path in SWEEP_IMAGES:
    name = os.path.basename(img_path)
    print(f'\n{"─"*55}')
    print(f'  Image: {name}')
    print(f'{"─"*55}')
    print(f'  {"k":>5}  {"τ":>7}  {"early%":>7}  {"PSNR":>7}  {"SSIM":>6}  {"time":>5}')
    sweep_results[name] = []

    for k in MULTIPLIERS:
        r = encode_with_multiplier(img_path, k)
        sweep_results[name].append(r)
        print(f'  {k:>5.2f}  {r["threshold"]:>7.1f}  '
              f'{r["early_stop_pct"]:>6.1f}%  '
              f'{r["psnr"]:>7.2f}  {r["ssim"]:>6.4f}  '
              f'{r["encode_time_s"]:>4.1f}s'
              + ('  ← current' if k == 0.50 else ''))

with open('/content/results_threshold_sweep.json', 'w') as f:
    json.dump(sweep_results, f, indent=2)

# ── print the key conclusion ──────────────────────────────────────────────
print(f'\n{"="*55}')
print('  THRESHOLD SWEEP SUMMARY')
print(f'{"="*55}')
for name, rows in sweep_results.items():
    psnr_at_05 = next(r['psnr'] for r in rows if r['k'] == 0.50)
    time_at_05 = next(r['encode_time_s'] for r in rows if r['k'] == 0.50)
    psnr_at_08 = next(r['psnr'] for r in rows if r['k'] == 0.80)
    time_at_01 = next(r['encode_time_s'] for r in rows if r['k'] == 0.10)
    print(f'  {name}:')
    print(f'    k=0.10 (tight): {time_at_01:.1f}s encode')
    print(f'    k=0.50 (used):  {time_at_05:.1f}s encode  {psnr_at_05:.2f} dB')
    print(f'    k=0.80 (loose): PSNR={psnr_at_08:.2f} dB '
          f'(Δ={psnr_at_08 - psnr_at_05:+.2f} dB vs k=0.50)')

print('\nSaved: /content/results_threshold_sweep.json')


## D3 — Residual Gate Justification

For each Kodak image, encodes twice (with and without residual pass) and measures
ΔPSNR = PSNR_with_residual − PSNR_without, grouped by early-stop rate zone.

**Expected result:**
- `< 40%`: ΔPSNR ≈ 0 or negative — diffuse gradient, residual unhelpful
- `40–70%`: ΔPSNR > 0 — structured residual, second pass improves quality
- `> 70%`: ΔPSNR ≈ 0 or negative — noise floor, residual adds artifacts

Run this after Cell D1 (requires Kodak images to be downloaded). Results saved
to `results_residual_gate.json`. The summary table gives empirically derived
gate boundaries from data rather than from assertion.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL F — RESIDUAL GATE JUSTIFICATION
# ║
# ║  PURPOSE: justify the 40-70% gate by measuring ΔPSNR from adding a
# ║           residual pass across ALL Kodak images, grouped by early-stop %.
# ║
# ║  METHOD:  for each image, encode twice:
# ║           (a) pass 1 only
# ║           (b) pass 1 + residual pass 2 (regardless of gate)
# ║           Record ΔPSNR = PSNR_with_residual − PSNR_without
# ║           Plot against early-stop rate.
# ║
# ║  EXPECTED RESULT:
# ║           <40%:  ΔPSNR ≈ 0 or negative (attractor mismatch, residual unhelpful)
# ║           40-70%: ΔPSNR > 0 (structured residual, improvable)
# ║           >70%:  ΔPSNR ≈ 0 or negative (noise floor, residual adds artifacts)
# ║
# ║  OUTPUT:  results_residual_gate.json
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, json, time
import numpy as np
from PIL import Image


def encode_and_measure(image_path, use_residual, time_budget=90):
    """
    Encode image, optionally with residual pass, return quality metrics.
    use_residual=True forces residual regardless of gate.
    use_residual=False skips it entirely.
    """
    img_raw = np.array(Image.open(image_path).convert('RGB'))
    img, orig_H, orig_W = pad_to_block_multiple(img_raw)
    H_pad, W_pad = img.shape[:2]
    ycbcr = rgb_to_ycbcr(img)

    cfg = analyse_image(ycbcr[:,:,0], time_budget, auto_threshold=True)
    cfg['colour_mode'] = 'LOSSY'
    sds, smeta = build_domain_stack(ycbcr[:,:,0].astype(np.float32), cfg['domain_step'])
    cfg = run_presample_diagnostic(ycbcr[:,:,0], sds, cfg, 0.05)
    cfg['colour_mode'] = 'LOSSY'

    y_tf, actual_early_pct = encode_channel_gpu(ycbcr[:,:,0], cfg)

    # ── pass 1 only quality ───────────────────────────────────────────────
    y_dec1 = decode_fractal(y_tf, (H_pad, W_pad), N_ITERATIONS)
    cb_enc = encode_colour_channel(ycbcr[:,:,1], 'LOSSY', DCT_QUALITY_FACTOR)
    cr_enc = encode_colour_channel(ycbcr[:,:,2], 'LOSSY', DCT_QUALITY_FACTOR)
    cb_dec = decode_colour_channel(cb_enc)
    cr_dec = decode_colour_channel(cr_enc)

    def ycbcr_to_rgb_dec(y):
        yd = np.stack([y, cb_dec, cr_dec], axis=2)
        return np.clip(ycbcr_to_rgb(yd), 0, 255).astype(np.uint8)[:orig_H, :orig_W]

    def psnr_ssim(dec):
        orig_f = img_raw[:orig_H, :orig_W].astype(np.float32)
        dec_f  = dec.astype(np.float32)
        mse    = float(np.mean((orig_f - dec_f)**2))
        psnr   = float(10 * np.log10(255**2 / mse)) if mse > 0 else 100.0
        ssim   = float(_ssim(orig_f, dec_f))
        return round(psnr, 3), round(ssim, 4)

    psnr1, ssim1 = psnr_ssim(ycbcr_to_rgb_dec(y_dec1))

    # ── pass 2 (forced regardless of gate) ───────────────────────────────
    if use_residual:
        y2_tf   = encode_residual(ycbcr[:,:,0], y_dec1, cfg)
        y_dec2  = decode_with_residual(y_tf, y2_tf, (H_pad, W_pad), N_ITERATIONS)
        psnr2, ssim2 = psnr_ssim(ycbcr_to_rgb_dec(y_dec2))
    else:
        psnr2, ssim2 = psnr1, ssim1

    return {
        'image':            os.path.basename(image_path),
        'early_stop_pct':   round(actual_early_pct * 100, 1),
        'psnr_pass1':       psnr1,
        'ssim_pass1':       ssim1,
        'psnr_pass2':       psnr2,
        'ssim_pass2':       ssim2,
        'delta_psnr':       round(psnr2 - psnr1, 3),
        'delta_ssim':       round(ssim2 - ssim1, 4),
        'in_gate':          0.40 <= actual_early_pct <= 0.70,
        'median_var':       round(cfg.get('median_var', 0), 1),
    }


# ── run on all available Kodak images ────────────────────────────────────
kodak_dir = '/content/kodak'
image_paths = sorted([
    os.path.join(kodak_dir, f)
    for f in os.listdir(kodak_dir)
    if f.endswith('.png')
])

if not image_paths:
    print('ERROR: No Kodak images found. Run Cell D first to download them.')
else:
    gate_results = []
    print(f'{"="*65}')
    print(f'  RESIDUAL GATE ANALYSIS ({len(image_paths)} images)')
    print(f'{"="*65}')
    print(f'  {"image":<16} {"early%":>7}  {"PSNR1":>7}  '
          f'{"PSNR2":>7}  {"ΔPSNR":>7}  {"gate":>5}')

    for path in image_paths:
        r = encode_and_measure(path, use_residual=True)
        gate_results.append(r)
        flag = ' ✓' if r['in_gate'] else '  '
        print(f'  {r["image"]:<16} {r["early_stop_pct"]:>6.1f}%  '
              f'{r["psnr_pass1"]:>7.3f}  {r["psnr_pass2"]:>7.3f}  '
              f'{r["delta_psnr"]:>+7.3f}{flag}')

    with open('/content/results_residual_gate.json', 'w') as f:
        json.dump(gate_results, f, indent=2)

    # ── summarise by zone ─────────────────────────────────────────────────
    below = [r for r in gate_results if r['early_stop_pct'] <  40]
    gate  = [r for r in gate_results if 40 <= r['early_stop_pct'] <= 70]
    above = [r for r in gate_results if r['early_stop_pct'] >  70]

    print(f'\n{"="*65}')
    print(f'  SUMMARY BY ZONE')
    print(f'{"="*65}')
    for zone_name, zone in [('<40% (below gate)', below),
                             ('40-70% (in gate)', gate),
                             ('>70% (above gate)', above)]:
        if zone:
            deltas = [r['delta_psnr'] for r in zone]
            print(f'  {zone_name}: {len(zone)} images  '
                  f'mean ΔPSNR={np.mean(deltas):+.3f} dB  '
                  f'min={min(deltas):+.3f}  max={max(deltas):+.3f}')
        else:
            print(f'  {zone_name}: 0 images (no images fell in this range)')

    # ── find the stable gate boundaries from data ─────────────────────────
    print(f'\n  EMPIRICALLY DERIVED GATE from this dataset:')
    # find percentile range where mean ΔPSNR > 0.1 dB
    sorted_by_rate = sorted(gate_results, key=lambda r: r['early_stop_pct'])
    print(f'  {"early%":>7}  {"ΔPSNR":>8}  note')
    for r in sorted_by_rate:
        note = ' ← gate used' if r['in_gate'] else ''
        note += ' ← crossover' if abs(r['delta_psnr']) < 0.05 else ''
        print(f'  {r["early_stop_pct"]:>6.1f}%  {r["delta_psnr"]:>+8.3f} dB{note}')

    print(f'\nSaved: /content/results_residual_gate.json')
    print('These results directly justify (or refine) the 40-70% gate in the paper.')


## D4 — Dataset Upscaling Comparison (Kodak subset)

Runs the same round-trip upscaling evaluation as Cell C1 across multiple Kodak images.
For each image and scale (2×, 4×, 8×), fractal native decode is compared against
bicubic and Lanczos interpolation from JPEG at matched file size.

All baselines use the same round-trip methodology: upscale → area-downscale back to
original resolution → measure against original. Incremental JSON save so a crash
does not lose progress.

**Prerequisites:** Kodak images and .frac files from Cell D1, plus results_kodak.json.

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════╗
# ║  CELL D4 — DATASET UPSCALING COMPARISON (Kodak subset)               ║
# ║                                                                        ║
# ║  For each image in the Kodak dataset that has a pre-encoded .frac     ║
# ║  file, run the same round-trip upscaling comparison used in Cell C1   ║
# ║  (fractal native decode vs bicubic/Lanczos from JPEG at matched KB)  ║
# ║  at scales 2×, 4×, 8×.                                               ║
# ║                                                                        ║
# ║  Round-trip methodology (same as C1):                                 ║
# ║    1. Each method produces a scale× output                            ║
# ║    2. Area-downscale back to original resolution                      ║
# ║    3. PSNR/SSIM measured against the original                        ║
# ║  No algorithm is used as its own reference. All methods at equal      ║
# ║  storage budget matched to fractal file size.                         ║
# ║                                                                        ║
# ║  PREREQUISITES:                                                        ║
# ║    - Kodak images in KODAK_DIR (downloaded by Cell D1)               ║
# ║    - Corresponding .frac files in FRAC_DIR (produced by Cell D1)     ║
# ║    - Cell D1 results in results_kodak.json                           ║
# ╚════════════════════════════════════════════════════════════════════════╝

import os, json, time
import numpy as np
import cv2
from PIL import Image as PILImage
from datetime import datetime

try:
    from skimage.metrics import structural_similarity as ssim_fn
except ImportError:
    import subprocess; subprocess.run(['pip','install','scikit-image','-q'])
    from skimage.metrics import structural_similarity as ssim_fn

# ── configuration ────────────────────────────────────────────────────────
KODAK_DIR   = '/content/kodak'          # directory with kodimXX.png
FRAC_DIR    = '/content/kodak_frac'     # directory with kodimXX.frac
KODAK_JSON  = '/content/results_kodak.json'   # output from Cell D1
SCALES      = [2, 4, 8]
LOG_PATH    = '/content/results_kodak_upscale.json'

# Subset to run — use None to run all available, or list filenames
# Running all 21 × 3 scales takes ~40 minutes on T4 due to decode cost
SUBSET = None   # e.g. ['kodim01.png', 'kodim05.png', 'kodim23.png']

MAX_IMAGES  = 8   # cap for reasonable runtime; set None for all
# ─────────────────────────────────────────────────────────────────────────


def img_metrics_upscale(orig_rgb, recon_rgb):
    o = orig_rgb.astype(np.float64)
    r = recon_rgb.astype(np.float64)
    mse = np.mean((o - r) ** 2)
    if mse == 0:
        return float('inf'), 1.0
    psnr = 10 * np.log10(255.0 ** 2 / mse)
    og = cv2.cvtColor(orig_rgb,  cv2.COLOR_RGB2GRAY)
    rg = cv2.cvtColor(recon_rgb, cv2.COLOR_RGB2GRAY)
    s  = ssim_fn(og, rg, data_range=255)
    return round(psnr, 2), round(float(s), 4)


def area_downscale(img_rgb, scale):
    H, W = img_rgb.shape[:2]
    return cv2.resize(img_rgb, (W // scale, H // scale),
                      interpolation=cv2.INTER_AREA)


def find_jpeg_quality_for_kb(img_rgb, target_kb):
    best_q, best_diff = 1, float('inf')
    for q in range(1, 96, 2):
        bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        diff = abs(len(buf) / 1024.0 - target_kb)
        if diff < best_diff:
            best_diff, best_q = diff, q
    return best_q


def upscale_interpolation_baseline(orig_rgb, scale, target_kb, interp_flag):
    """
    Round-trip baseline matching Cell C1 methodology exactly:
      1. Downscale to 1/scale using area averaging
      2. JPEG encode at quality matched to target_kb
      3. Upscale to original resolution with interp_flag
      4. Area-downscale back to original (round-trip step)
      5. Measure against original
    """
    H, W  = orig_rgb.shape[:2]
    small = area_downscale(orig_rgb, scale)
    q     = find_jpeg_quality_for_kb(small, target_kb)
    bgr   = cv2.cvtColor(small, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
    small_dec = cv2.cvtColor(
        cv2.imdecode(np.frombuffer(buf, np.uint8), cv2.IMREAD_COLOR),
        cv2.COLOR_BGR2RGB)
    kb = len(buf) / 1024.0
    upscaled  = cv2.resize(small_dec, (W, H), interpolation=interp_flag)
    roundtrip = cv2.resize(upscaled,  (W // scale, H // scale),
                           interpolation=cv2.INTER_AREA)
    roundtrip = cv2.resize(roundtrip, (W, H), interpolation=cv2.INTER_AREA)
    return roundtrip, round(kb, 1), q


# ── load Kodak results to get per-image frac_kb values ───────────────────
if os.path.exists(KODAK_JSON):
    with open(KODAK_JSON) as f:
        kodak_meta = {r['image']: r for r in json.load(f)}
else:
    print(f'WARNING: {KODAK_JSON} not found. '
          f'Run Cell D1 first, or set frac_kb manually.')
    kodak_meta = {}

# ── build file list ────────────────────────────────────────────────────────
if SUBSET:
    candidates = SUBSET
else:
    candidates = sorted([
        f for f in os.listdir(KODAK_DIR)
        if f.endswith('.png') and
           os.path.exists(os.path.join(FRAC_DIR, f.replace('.png', '.frac')))
    ]) if os.path.isdir(KODAK_DIR) else []

if MAX_IMAGES:
    candidates = candidates[:MAX_IMAGES]

print('=' * 72)
print('  KODAK UPSCALING COMPARISON — ROUND-TRIP EVALUATION')
print(f'  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Images: {len(candidates)}   Scales: {SCALES}')
print('=' * 72)

all_results = {}
aggregate = {s: {'fractal':[], 'bicubic':[], 'lanczos':[], 'delta':[]}
             for s in SCALES}

for img_name in candidates:
    orig_path = os.path.join(KODAK_DIR, img_name)
    frac_path = os.path.join(FRAC_DIR, img_name.replace('.png', '.frac'))

    if not os.path.exists(orig_path):
        print(f'\n  [SKIP] {img_name} — original not found'); continue
    if not os.path.exists(frac_path):
        print(f'\n  [SKIP] {img_name} — .frac not found'); continue

    meta     = kodak_meta.get(img_name, {})
    frac_kb  = meta.get('frac_kb', None)
    if frac_kb is None:
        frac_kb = os.path.getsize(frac_path) / 1024.0
        print(f'  (using on-disk size {frac_kb:.1f} KB for {img_name})')

    orig_rgb = np.array(PILImage.open(orig_path).convert('RGB'))
    H, W     = orig_rgb.shape[:2]

    print(f'\n  ══ {img_name}  ({W}×{H})  fractal: {frac_kb:.1f} KB ══')

    # Load .frac once — reused across all scales
    t_load = time.time()
    try:
        padded_shape, (orig_H, orig_W), y_tf, cb_enc, cr_enc, fcfg, y2_tf = \
            load_compressed(frac_path)
    except Exception as e:
        print(f'  [ERROR] Could not load {frac_path}: {e}'); continue
    print(f'  Loaded in {time.time()-t_load:.1f}s')

    image_results = {}

    for scale in SCALES:
        sH, sW = H * scale, W * scale
        row    = {}

        # ── 1. Fractal native decode ────────────────────────────────────
        t0 = time.time()
        frac_up = decode_and_upscale(
            y_tf, cb_enc, cr_enc,
            padded_shape, (orig_H, orig_W),
            scale=scale, n_i=N_ITERATIONS + 2
        )
        frac_up  = frac_up[:sH, :sW]
        frac_rt  = cv2.resize(frac_up, (W, H), interpolation=cv2.INTER_AREA)
        f_psnr, f_ssim = img_metrics_upscale(orig_rgb, frac_rt)
        row['fractal'] = {'psnr': f_psnr, 'ssim': f_ssim, 'kb': round(frac_kb,1)}
        t_frac = time.time() - t0

        # ── 2. Bicubic baseline ─────────────────────────────────────────
        bic_rt, bic_kb, bic_q = upscale_interpolation_baseline(
            orig_rgb, scale, frac_kb, cv2.INTER_CUBIC)
        b_psnr, b_ssim = img_metrics_upscale(orig_rgb, bic_rt)
        row['bicubic'] = {'psnr': b_psnr, 'ssim': b_ssim,
                          'kb': bic_kb, 'jpeg_q': bic_q}

        # ── 3. Lanczos baseline ─────────────────────────────────────────
        lz_rt, lz_kb, lz_q = upscale_interpolation_baseline(
            orig_rgb, scale, frac_kb, cv2.INTER_LANCZOS4)
        l_psnr, l_ssim = img_metrics_upscale(orig_rgb, lz_rt)
        row['lanczos'] = {'psnr': l_psnr, 'ssim': l_ssim,
                          'kb': lz_kb,  'jpeg_q': lz_q}

        best_baseline = max(b_psnr, l_psnr)
        delta = round(f_psnr - best_baseline, 2)
        verdict = '◀ FRACTAL' if delta > 0 else '  baseline'

        row['delta_psnr_vs_baseline'] = delta
        row['best_baseline_name']     = 'bicubic' if b_psnr >= l_psnr else 'lanczos'
        image_results[f'{scale}x']    = row

        aggregate[scale]['fractal'].append(f_psnr)
        aggregate[scale]['bicubic'].append(b_psnr)
        aggregate[scale]['lanczos'].append(l_psnr)
        aggregate[scale]['delta'].append(delta)

        print(f'  {scale}×  fractal={f_psnr:>7.2f}  bicubic={b_psnr:>7.2f}  '
              f'lanczos={l_psnr:>7.2f}  Δ={delta:>+6.2f} dB  {verdict}'
              f'  ({t_frac:.0f}s decode)')

    all_results[img_name] = image_results

    # Incremental save
    with open(LOG_PATH, 'w') as f:
        json.dump({'per_image': all_results, 'aggregate': aggregate}, f, indent=2)

# ── aggregate summary ──────────────────────────────────────────────────────
print()
print('=' * 72)
print('  AGGREGATE SUMMARY — mean PSNR across all evaluated images')
print('=' * 72)
print(f'  {"Scale":>5}  {"Fractal":>9}  {"Bicubic":>9}  {"Lanczos":>9}  '
      f'{"Mean Δ":>9}  {"Fractal wins":>13}')
print('  ' + '─' * 65)
for scale in SCALES:
    ag = aggregate[scale]
    if not ag['fractal']: continue
    n   = len(ag['fractal'])
    mf  = sum(ag['fractal']) / n
    mb  = sum(ag['bicubic']) / n
    ml  = sum(ag['lanczos']) / n
    md  = sum(ag['delta'])   / n
    wins = sum(1 for d in ag['delta'] if d > 0)
    print(f'  {scale}×  {mf:>9.2f}  {mb:>9.2f}  {ml:>9.2f}  '
          f'{md:>+9.2f}  {wins}/{n} images')

print()
print('  Round-trip: upscaled output area-downscaled to original resolution')
print('  All baselines matched to fractal file size. No method self-referenced.')
print(f'\n  Full results saved → {LOG_PATH}')


# Sample Generation
generates the samples needed for the web app once and the subsequent jpeg values and versions

In [21]:
"""
fractal_core.py — decode-only runtime for fractal image viewer.

Extracted from fractal_compress_v7_3.ipynb (cells A4–A13).
No GPU required. No Colab dependencies. Pure numpy + scipy + PIL.

Usage:
    from fractal_core import load_compressed, decode_and_upscale
    padded_shape, orig_shape, y_tf, cb_enc, cr_enc, cfg, _ = load_compressed("image.frac")
    rgb = decode_and_upscale(y_tf, cb_enc, cr_enc, padded_shape, orig_shape, scale=4)
"""

import math
import struct
import zlib

import numpy as np
from PIL import Image
from scipy.fft import idctn

# ── Constants (must match encoder) ────────────────────────────────────────
BLOCK_SIZE     = 8
DOMAIN_SIZE    = 16
NUM_TRANSFORMS = 8
N_ITERATIONS   = 10

# Standard JPEG chroma quantisation table (used by DCT colour encoder)
CHROMA_QUANT_BASE = np.array([
    [16, 11, 10, 16,  24,  40,  51,  61],
    [12, 12, 14, 19,  26,  58,  60,  55],
    [14, 13, 16, 24,  40,  57,  69,  56],
    [14, 17, 22, 29,  51,  87,  80,  62],
    [18, 22, 37, 56,  68, 109, 103,  77],
    [24, 35, 55, 64,  81, 104, 113,  92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103,  99],
], dtype=np.float32)

MAGIC   = b'FRAC'
VERSION = 7


# ── Colour space ──────────────────────────────────────────────────────────

def ycbcr_to_rgb(image):
    img = image.astype(np.float32)
    img[:, :, 1] -= 128
    img[:, :, 2] -= 128
    inv = np.array([
        [1.000,  0.000,  1.402],
        [1.000, -0.344, -0.714],
        [1.000,  1.772,  0.000],
    ])
    return np.clip(img @ inv.T, 0, 255).astype(np.uint8)


# ── Fractal helpers ───────────────────────────────────────────────────────

def downsample_2x(block):
    h, w = block.shape
    return block.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3)).astype(np.float32)


# ── Colour decoder ────────────────────────────────────────────────────────

def _zigzag_order(n=8):
    idx = []
    for d in range(2 * n - 1):
        if d % 2 == 0:
            r = min(d, n - 1); c = d - r
            while r >= 0 and c < n:
                idx.append((r, c)); r -= 1; c += 1
        else:
            c = min(d, n - 1); r = d - c
            while c >= 0 and r < n:
                idx.append((r, c)); r += 1; c -= 1
    return idx

_ZZ = _zigzag_order()


def decode_channel_dct_compact(compressed, quant_table, original_shape):
    H, W = original_shape
    data = zlib.decompress(compressed)
    out  = np.zeros((H, W), dtype=np.float32)
    i    = 0
    for ry in range(0, H - BLOCK_SIZE + 1, BLOCK_SIZE):
        for rx in range(0, W - BLOCK_SIZE + 1, BLOCK_SIZE):
            last_nz = data[i]; i += 1
            zz = np.zeros(64, dtype=np.float32)
            if last_nz > 0:
                vals = np.frombuffer(
                    data[i:i + last_nz * 2], dtype=np.int16
                ).astype(np.float32)
                zz[:last_nz] = vals
                i += last_nz * 2
            grid = np.zeros((8, 8), dtype=np.float32)
            for k, (r, c) in enumerate(_ZZ):
                grid[r, c] = zz[k] * quant_table[r, c]
            out[ry:ry + BLOCK_SIZE, rx:rx + BLOCK_SIZE] = (
                idctn(grid, norm='ortho') + 128
            )
    return np.clip(out, 0, 255).astype(np.uint8)


def _delta_decode(d):
    s = d.astype(np.int32)
    s[:, 1:] = np.where(
        d[:, 1:] > 127,
        d[:, 1:].astype(np.int32) - 256,
        d[:, 1:].astype(np.int32),
    )
    return np.clip(np.cumsum(s, axis=1) % 256, 0, 255).astype(np.uint8)


def decode_channel_lossless(comp_bytes, original_shape):
    H, W  = original_shape
    delta = np.frombuffer(
        zlib.decompress(comp_bytes), dtype=np.uint8
    ).reshape(H, W)
    return _delta_decode(delta)


def decode_colour_channel(enc):
    if enc['mode'] == 'LOSSLESS':
        return decode_channel_lossless(enc['data'], enc['shape'])
    else:
        return decode_channel_dct_compact(enc['comp'], enc['quant'], enc['shape'])


# ── Fractal decoder ───────────────────────────────────────────────────────

def decode_fractal_upscaled(transforms, original_shape, scale=1, n_iter=None):
    """
    Decode the stored IFS transforms at `scale`× resolution.

    The same .frac file decodes at any integer scale by applying the
    stored rules on a proportionally larger canvas. Quality converges
    to a stable floor at 4× (confirmed across all Kodak-24 images).

    Args:
        transforms:     list of (dy, dx, ti, c, b) tuples from load_compressed
        original_shape: (H, W) of the padded encoded image
        scale:          integer upscale factor (1, 2, 4, 8, 10 …)
        n_iter:         IFS iterations; defaults to N_ITERATIONS + 2 for scale > 1

    Returns:
        uint8 numpy array of shape (H*scale, W*scale)
    """
    if n_iter is None:
        n_iter = N_ITERATIONS if scale == 1 else N_ITERATIONS + 2

    ratio   = DOMAIN_SIZE // BLOCK_SIZE
    H, W    = original_shape
    sH, sW  = H * scale, W * scale
    sB, sD  = BLOCK_SIZE * scale, DOMAIN_SIZE * scale

    current = np.full((sH, sW), 128.0, dtype=np.float32)
    pos = [
        (ry, rx)
        for ry in range(0, H - BLOCK_SIZE + 1, BLOCK_SIZE)
        for rx in range(0, W - BLOCK_SIZE + 1, BLOCK_SIZE)
    ]

    for _ in range(n_iter):
        nxt = np.zeros((sH, sW), dtype=np.float32)
        for bi, (ry, rx) in enumerate(pos):
            dy, dx, ti, c, b = transforms[bi]
            sry, srx = ry * scale, rx * scale
            sdy, sdx = dy * scale, dx * scale
            raw = current[sdy:sdy + sD, sdx:sdx + sD]
            if raw.shape[0] < sD or raw.shape[1] < sD:
                nxt[sry:sry + sB, srx:srx + sB] = b
                continue
            domain = raw.reshape(sB, ratio, sB, ratio).mean(axis=(1, 3))
            if ti >= 4:
                domain = np.fliplr(domain)
            domain = np.rot90(domain, ti % 4)
            nxt[sry:sry + sB, srx:srx + sB] = c * domain + b
        current = nxt

    return np.clip(current, 0, 255).astype(np.uint8)


def decode_and_upscale(y_tf, cb_enc, cr_enc, padded_shape, orig_shape, scale=1):
    """
    Full pipeline: decode Y channel at scale×, resize chroma, convert to RGB,
    crop padding introduced during encoding.

    Returns:
        uint8 RGB numpy array of shape (orig_H*scale, orig_W*scale, 3)
    """
    H, W   = padded_shape
    oH, oW = orig_shape

    y_up = decode_fractal_upscaled(y_tf, (H, W), scale=scale)
    cb   = decode_colour_channel(cb_enc)
    cr   = decode_colour_channel(cr_enc)

    cb_up = np.array(
        Image.fromarray(cb).resize((W * scale, H * scale), Image.BILINEAR)
    )
    cr_up = np.array(
        Image.fromarray(cr).resize((W * scale, H * scale), Image.BILINEAR)
    )

    rgb = ycbcr_to_rgb(np.stack([y_up, cb_up, cr_up], axis=2))
    return rgb[:oH * scale, :oW * scale]


# ── File loader ───────────────────────────────────────────────────────────

def _unpack_transform(data):
    p  = struct.unpack('>I', data)[0]
    cy = (p >> 26) & 0x3F
    cx = (p >> 20) & 0x3F
    ti = (p >> 17) & 0x07
    cq = ((p >> 9) & 0xFF) - 127
    bq = (p >> 1) & 0xFF
    return cy, cx, ti, cq, bq


def load_compressed(filepath):
    """
    Load a .frac v7 file.

    Returns:
        padded_shape : (H, W) — padded dimensions used during encoding
        orig_shape   : (oH, oW) — original image dimensions before padding
        y_transforms : list of (dy, dx, ti, c, b) tuples
        cb_enc       : colour channel dict for Cb
        cr_enc       : colour channel dict for Cr
        cfg          : dict with domain_step and colour_mode
        y2_transforms: residual pass transforms or None
    """
    import pickle

    with open(filepath, 'rb') as f:
        magic = f.read(4)
        assert magic == MAGIC, f'Not a .frac file (bad magic: {magic})'

        version = struct.unpack('I', f.read(4))[0]
        H, W, oH, oW = struct.unpack('4I', f.read(16))
        n_y    = struct.unpack('I', f.read(4))[0]
        ds     = struct.unpack('I', f.read(4))[0]
        mode_b = struct.unpack('B', f.read(1))[0]
        has_r  = struct.unpack('B', f.read(1))[0]
        n_cb   = struct.unpack('I', f.read(4))[0]
        n_cr   = struct.unpack('I', f.read(4))[0]

        y_transforms = []
        for _ in range(n_y):
            cy, cx, ti, cq, bq = _unpack_transform(f.read(4))
            y_transforms.append((cy * ds, cx * ds, ti, cq, bq))

        y2_transforms = None
        if has_r:
            n_y2 = struct.unpack('I', f.read(4))[0]
            y2_transforms = []
            for _ in range(n_y2):
                cy, cx, ti, cq, bq = _unpack_transform(f.read(4))
                y2_transforms.append((cy * ds, cx * ds, ti, cq, bq))

        cb_blob = f.read(n_cb)
        cr_blob = f.read(n_cr)

    colour_mode = 'LOSSLESS' if mode_b else 'LOSSY'

    if colour_mode == 'LOSSLESS':
        cb_enc = {'mode': 'LOSSLESS', 'data': cb_blob, 'shape': (H, W)}
        cr_enc = {'mode': 'LOSSLESS', 'data': cr_blob, 'shape': (H, W)}
    else:
        cbd = pickle.loads(cb_blob)
        crd = pickle.loads(cr_blob)
        cb_enc = {'mode': 'LOSSY', 'comp': cbd['comp'],
                  'quant': cbd['quant'], 'shape': cbd['shape']}
        cr_enc = {'mode': 'LOSSY', 'comp': crd['comp'],
                  'quant': crd['quant'], 'shape': crd['shape']}

    cfg = {
        'domain_step':  ds,
        'colour_mode':  colour_mode,
        'error_threshold': 0,
        'flat_means': {}, 'flat_mask': [],
        'flat_pct': 0, 'median_var': 0,
        'total_blocks': 0, 'range_vars': [],
    }

    return (H, W), (oH, oW), y_transforms, cb_enc, cr_enc, cfg, y2_transforms


In [23]:
"""
prepare_samples.py — run ONCE on Colab (GPU required for encoding).

For each source image this script:
  1. Encodes it → .frac file using your existing notebook pipeline
  2. Generates JPEG versions at thumbnail / medium / full resolution
     at quality levels that match the .frac file size as closely as possible
  3. Writes a meta.json per sample with file sizes and labels

Output folder structure (copy the whole `samples/` folder next to app.py):

    samples/
        webpage/
            label.txt           human-readable name shown in the app
            original.png        source image
            encoded.frac        fractal compressed
            jpeg_thumb.jpg      JPEG at 1/8 res  (as served for thumbnails)
            jpeg_medium.jpg     JPEG at 1/4 res  (as served for previews)
            jpeg_full.jpg       JPEG at 1/1 res  (as served for full view)
            meta.json           sizes + labels
        document/
            ...
        kodak_landscape/
            ...

HOW TO USE:
    1. Open your fractal_compress_v7_3.ipynb in Colab (T4 GPU).
    2. Run all A-section cells so all functions are defined.
    3. Paste and run this script in a new cell at the bottom, or upload
       it and run:  exec(open('prepare_samples.py').read())
    4. Download the generated `samples/` folder to run alongside app.py.

CONFIGURE the SOURCE_IMAGES list below before running.
"""

import os
import json
import numpy as np
import cv2
from PIL import Image as PILImage

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
# Add your own images here. 'path' is the Colab path to the source image.
# 'label' is what the viewer will display.
SOURCE_IMAGES = [
    {
        "key":   "webpage",
        "path":  "/content/portrait showcase.png",   # ← set your path
        "label": "Webpage Assets",
        "description": "Product images and UI assets served at multiple sizes",
    },
    {
        "key":   "document",
        "path":  "/content/hall way.png",  # ← set your path
        "label": "Document Archive",
        "description": "Scanned documents stored once, viewed at any zoom",
    },
    {
        "key":   "kodak_landscape",
        "path":  "/content/horizon.jpg",     # ← Kodak benchmark image
        "label": "High-Res Photo",
        "description": "Photography archive — single file, any resolution",
    },
]

OUTPUT_DIR = "/content/samples"  # will be created if it doesn't exist

# Resolutions to pre-generate as JPEG (fractions of original)
JPEG_TIERS = {
    "thumb":  8,   # 1/8  resolution — thumbnail
    "medium": 4,   # 1/4  resolution — preview
    "full":   1,   # 1/1  resolution — full size
}
# ─────────────────────────────────────────────────────────────────────────


def find_jpeg_quality(img_rgb, target_kb, tolerance_kb=2.0):
    """Find JPEG quality whose output is closest to target_kb."""
    best_q, best_diff = 95, float('inf')
    for q in range(2, 96, 2):
        bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        _, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        diff = abs(len(buf) / 1024.0 - target_kb)
        if diff < best_diff:
            best_diff, best_q = diff, q
    return best_q


def generate_jpeg_tiers(orig_rgb, frac_kb, out_dir):
    """
    For each tier, downscale the image and save a JPEG.
    Quality is chosen so the JPEG matches frac_kb / len(tiers) each —
    i.e. if you were storing all tiers, total JPEG cost ≈ 3× frac cost.
    We do NOT size-match the JPEGs to the .frac — the point is to show
    that the JPEG approach requires multiple files.
    """
    H, W     = orig_rgb.shape[:2]
    # Use a natural quality (85) for all tiers — this is what a real CDN
    # would serve. The demo shows total size vs one .frac.
    sizes    = {}
    for tier_name, divisor in JPEG_TIERS.items():
        new_w = max(1, W // divisor)
        new_h = max(1, H // divisor)
        resized = cv2.resize(
            cv2.cvtColor(orig_rgb, cv2.COLOR_RGB2BGR),
            (new_w, new_h),
            interpolation=cv2.INTER_AREA,
        )
        out_path = os.path.join(out_dir, f"jpeg_{tier_name}.jpg")
        cv2.imwrite(out_path, resized, [cv2.IMWRITE_JPEG_QUALITY, 85])
        kb = os.path.getsize(out_path) / 1024.0
        sizes[tier_name] = {
            "path":  out_path,
            "kb":    round(kb, 1),
            "res":   f"{new_w}×{new_h}",
            "scale": divisor,
        }
        print(f"    JPEG {tier_name:8s} ({new_w}×{new_h})  →  {kb:.1f} KB")
    return sizes


def prepare_sample(entry):
    key         = entry["key"]
    src_path    = entry["path"]
    label       = entry["label"]
    description = entry["description"]
    out_dir     = os.path.join(OUTPUT_DIR, key)
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.exists(src_path):
        print(f"  [SKIP] {key} — source not found: {src_path}")
        return

    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"  Source: {src_path}")

    # ── Load and save original ────────────────────────────────────────
    orig_rgb = np.array(PILImage.open(src_path).convert('RGB'))
    H, W     = orig_rgb.shape[:2]
    orig_out = os.path.join(out_dir, "original.png")
    PILImage.fromarray(orig_rgb).save(orig_out)
    orig_kb  = os.path.getsize(orig_out) / 1024.0
    print(f"  Original: {W}×{H}  ({orig_kb:.1f} KB as PNG)")

    # ── Fractal encode ────────────────────────────────────────────────
    # These calls use the functions defined in your notebook A-section.
    # Make sure all A-section cells have been run before calling this.
    frac_path = os.path.join(out_dir, "encoded.frac")
    print(f"  Encoding → {frac_path}")

    img_raw      = orig_rgb
    padded, oH, oW = pad_to_block_multiple(img_raw)        # from notebook
    ycbcr         = rgb_to_ycbcr(padded)                   # from notebook
    y_channel     = ycbcr[:, :, 0].astype(np.float32)
    cb_channel    = ycbcr[:, :, 1].astype(np.uint8)
    cr_channel    = ycbcr[:, :, 2].astype(np.uint8)

    cfg = analyse_image(y_channel)                          # from notebook
    domain_stack  = build_domain_stack(y_channel, cfg['domain_step'])  # notebook
    presample_ok  = run_presample_diagnostic(y_channel, domain_stack, cfg)  # notebook

    y_transforms  = encode_channel_gpu(y_channel, domain_stack, cfg)   # notebook
    cb_enc        = encode_colour_channel(cb_channel, 'LOSSY')          # notebook
    cr_enc        = encode_colour_channel(cr_channel, 'LOSSY')          # notebook

    save_compressed(
        frac_path,
        padded.shape[:2], (oH, oW),
        y_transforms, cb_enc, cr_enc, cfg,
    )

    frac_kb = os.path.getsize(frac_path) / 1024.0
    print(f"  Fractal file: {frac_kb:.1f} KB  "
          f"({100*(1 - frac_kb*1024/orig_rgb.nbytes):.1f}% compression)")

    # ── JPEG tiers ────────────────────────────────────────────────────
    print(f"  Generating JPEG tiers:")
    jpeg_sizes = generate_jpeg_tiers(orig_rgb, frac_kb, out_dir)
    total_jpeg_kb = sum(t["kb"] for t in jpeg_sizes.values())

    # ── Label file ───────────────────────────────────────────────────
    with open(os.path.join(out_dir, "label.txt"), "w") as f:
        f.write(label)

    # ── Meta JSON ────────────────────────────────────────────────────
    meta = {
        "key":         key,
        "label":       label,
        "description": description,
        "image_res":   f"{W}×{H}",
        "frac_kb":     round(frac_kb, 1),
        "orig_kb":     round(orig_kb, 1),
        "jpeg_tiers":  jpeg_sizes,
        "jpeg_total_kb": round(total_jpeg_kb, 1),
        "storage_saving_pct": round(
            100 * (total_jpeg_kb - frac_kb) / total_jpeg_kb, 1
        ),
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n  ── Storage summary ──────────────────────────")
    print(f"  Fractal (1 file):              {frac_kb:>7.1f} KB")
    print(f"  JPEG (3 files, thumb+med+full):{total_jpeg_kb:>7.1f} KB")
    print(f"  Saving:                       "
          f" {meta['storage_saving_pct']}% less storage with fractal")
    print(f"  Saved to: {out_dir}")


# ── Run ───────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

for entry in SOURCE_IMAGES:
    try:
        prepare_sample(entry)
    except Exception as e:
        print(f"\n  [ERROR] {entry['key']}: {e}")
        import traceback; traceback.print_exc()

print(f"\n{'='*60}")
print(f"  Done. Download the samples/ folder from: {OUTPUT_DIR}")
print(f"  Place it next to app.py before running the viewer.")



  Processing: Webpage Assets
  Source: /content/portrait showcase.png
  Original: 1920×1280  (2380.9 KB as PNG)
  Encoding → /content/samples/webpage/encoded.frac
  Analysing image...
  Flat blocks: 1913/38400 (5.0%)
  Median block variance: 1.1
  Auto ERROR_THRESHOLD:  5.0
  Auto DOMAIN_STEP: 32  (budget=120s)
    19,200 candidates
  Pre-sample: 1824 blocks stratified across 3 variance groups...

  [ERROR] webpage: 'tuple' object has no attribute 'sum'

  Processing: Document Archive
  Source: /content/hall way.png


Traceback (most recent call last):
  File "/tmp/ipykernel_608/3353605710.py", line 216, in <cell line: 0>
    prepare_sample(entry)
  File "/tmp/ipykernel_608/3353605710.py", line 161, in prepare_sample
    presample_ok  = run_presample_diagnostic(y_channel, domain_stack, cfg)  # notebook
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_608/47506806.py", line 30, in run_presample_diagnostic
    sum_d  = xp.sum(domain_stack, axis=1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/_math/sumprod.py", line 42, in sum
    return a.sum(axis, dtype, out, keepdims)
           ^^^^^
AttributeError: 'tuple' object has no attribute 'sum'


  Original: 1440×1920  (3747.3 KB as PNG)
  Encoding → /content/samples/document/encoded.frac
  Analysing image...
  Flat blocks: 2160/43200 (5.0%)
  Median block variance: 38.8
  Auto ERROR_THRESHOLD:  19.4
  Auto DOMAIN_STEP: 32  (budget=120s)
    21,600 candidates
  Pre-sample: 2052 blocks stratified across 3 variance groups...

  [ERROR] document: 'tuple' object has no attribute 'sum'

  Processing: High-Res Photo
  Source: /content/horizon.jpg


Traceback (most recent call last):
  File "/tmp/ipykernel_608/3353605710.py", line 216, in <cell line: 0>
    prepare_sample(entry)
  File "/tmp/ipykernel_608/3353605710.py", line 161, in prepare_sample
    presample_ok  = run_presample_diagnostic(y_channel, domain_stack, cfg)  # notebook
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_608/47506806.py", line 30, in run_presample_diagnostic
    sum_d  = xp.sum(domain_stack, axis=1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/_math/sumprod.py", line 42, in sum
    return a.sum(axis, dtype, out, keepdims)
           ^^^^^
AttributeError: 'tuple' object has no attribute 'sum'


  Original: 1280×720  (758.9 KB as PNG)
  Encoding → /content/samples/kodak_landscape/encoded.frac
  Analysing image...
  Flat blocks: 716/14400 (5.0%)
  Median block variance: 3.1
  Auto ERROR_THRESHOLD:  5.0
  Auto DOMAIN_STEP: 24  (budget=120s)
    12,720 candidates
  Pre-sample: 684 blocks stratified across 3 variance groups...

  [ERROR] kodak_landscape: 'tuple' object has no attribute 'sum'

  Done. Download the samples/ folder from: /content/samples
  Place it next to app.py before running the viewer.


Traceback (most recent call last):
  File "/tmp/ipykernel_608/3353605710.py", line 216, in <cell line: 0>
    prepare_sample(entry)
  File "/tmp/ipykernel_608/3353605710.py", line 161, in prepare_sample
    presample_ok  = run_presample_diagnostic(y_channel, domain_stack, cfg)  # notebook
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_608/47506806.py", line 30, in run_presample_diagnostic
    sum_d  = xp.sum(domain_stack, axis=1)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/cupy/_math/sumprod.py", line 42, in sum
    return a.sum(axis, dtype, out, keepdims)
           ^^^^^
AttributeError: 'tuple' object has no attribute 'sum'
